# NB13：代糖混合物模型 — PLS vs 古典最小平方（NNLS/CLS）

**食品分析實驗 週次 16｜銜接 W15（代糖前處理）與 W16 課堂 deck**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 檢查 **5×5 全因子設計**（阿斯巴甜 0–10% × 三氯蔗糖 0–10% w/w，赤藻糖醇補足）的正交性（`corr(Y1,Y2)≈0`）。
2. 用**分組交叉驗證**（`GroupKFold`／`LeaveOneGroupOut`，以 `sample_id` 分組）比較前處理，並用 **PLS1** 分別對兩種代糖建模。
3. 用**離網格盲樣（off-grid blind blends）**算出誠實的 **RMSEP**，並整理成 model report card。
4. 算出 **VIP** 並對照拉曼位移的官能基指認，理解**封閉性（closure）**問題。
5. 用**純物質參考光譜**做 **NNLS（非負最小平方，古典最小平方 CLS）**擬合，跟 PLS 比較優缺點。
6. 用低濃度端資料估計**近似 LOD**（教學簡化法，非正式方法確效）。
7. 用 **Hotelling T²／Q 殘差** 判斷一個市售樣品是否落在模型的適用範圍內，並誠實報告結果。

## 如何使用本筆記本

- 直接「全部執行」即可用內建的**模擬 Raman 785 光譜**跑完整份筆記本。
- 資料格式與共用函式（`load_spectra`／`split_spectra`／`plot_spectra`）與 NB09–NB12 一致，規範見下方 Setup 小節。

> ⚠️ 本筆記本的資料是**模擬教學資料**，不是真實量測值。

## 1. 環境設定 (Setup)

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
from scipy.optimize import nnls
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.model_selection import LeaveOneGroupOut, GroupKFold, KFold

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 1607
np.random.seed(RNG_SEED)

### 光譜資料格式 (Shared Spectral Data Format) — NB09–NB13 共用

從本週開始，資料改用**寬格式 (wide format)**，跟 NB01 的長格式不同：**一列＝一條完整光譜**。

| 規則 | 說明 |
|---|---|
| 中繼資料欄位在前 | 欄名**不是數字**的欄位，例如 `sample_id`（樣品代號）、`group`（組別 1–5，沿用 NB01 的定義；TA 提供的參考/示範資料用 `0`）、`instrument`（儀器代號）、`rep`（重複次數）、`role`（`library`/`unknown`/`mixture` 等角色）、以及依主題而定的標籤欄位，例如 `class`（化合物名稱）、`oil`（油品代號）、`acidity_pct`、`aspartame_pct`、`sucralose_pct` 等。 |
| 光譜欄位在後 | 欄名**能轉成浮點數**的欄位＝光譜資料本身，欄名即該點的波長 (nm，NIR) 或波數 (cm⁻¹，Raman)。程式一律依欄名數值排序，不假設檔案裡欄位順序已經由小到大。 |
| 光譜欄位不可缺值 | 每一列在所有光譜欄位都必須有數值，不可有 NaN。 |
| **一個檔案只用一種共同 x 軸** | Raman 位移與 NIR 波長不可混在同一個檔案；不同解析度/範圍的儀器資料也請分開存檔，或先內插到同一組 x 軸再合併。 |
| 標籤欄位的值用英文代碼 | 例如油品用 `EVOO`/`COC`，化合物用 `aspartame`/`sucralose`。畫圖的圖例文字直接取自欄位值，中文會因 Colab 字型問題變成方框。 |

**也相容 SpectraView 的「rows」匯出格式**（`specview/formats/__init__.py` 的 `save_combined_csv(..., layout="rows")`）：
第一列表頭的第一格是 `x:<軸標籤>`（例如 `x:Raman shift (cm-1)`），之後每一格才是 x 值本身；
從第二列開始，每一列的第一格是**光譜名稱**、其餘是強度值。這種格式沒有額外中繼資料欄位，
只有一個 `sample_id`——如果要加上組別/儀器等資訊，請自行用 `sample_id` 做 `merge`。

共用讀取/拆解/畫圖函式（簽名固定，NB09–NB13 都一樣）：

```python
load_spectra(source=None)          # -> 寬格式 DataFrame；None 時退回內建模擬光譜
split_spectra(df)                  # -> (meta, X, axis)
plot_spectra(axis, X, labels)      # -> matplotlib Axes，依 labels 分色、圖例自動去重
```

In [ ]:
def _is_number(text):
    try:
        float(text)
        return True
    except (TypeError, ValueError):
        return False


def load_spectra(source=None):
    """讀取寬格式光譜 CSV（見上方「光譜資料格式」說明）。

    Parameters
    ----------
    source : str, path, file-like, or None
        - None：退回本筆記本內嵌的模擬光譜字串 SAMPLE_CSV（模擬光譜，非真實量測）。
        - 本機/雲端硬碟路徑，或 http(s) URL（例如已發布的 Google 試算表 CSV 連結）：
          直接用 pandas.read_csv 讀取。

    Returns
    -------
    pandas.DataFrame
        寬格式：中繼資料欄位（欄名非數字）+ 光譜欄位（欄名為數字，代表 nm 或 cm-1）。
    """
    if source is None:
        return pd.read_csv(io.StringIO(SAMPLE_CSV))
    return pd.read_csv(source)


def split_spectra(df):
    """把 load_spectra() 讀進來的寬格式 DataFrame 拆成 (meta, X, axis)。

    自動判斷兩種輸入：
    - 一般格式：欄名可轉成浮點數的欄位＝光譜欄位；其餘欄位＝中繼資料 (meta)。
    - SpectraView「rows」匯出格式：第一個欄名以 "x:" 開頭，代表該欄其實是
      「樣品名稱」，其餘欄名本身就是 x 值；此時 meta 只有一欄 sample_id
      （由第一欄改名而來），其他中繼資料需自行用 sample_id 合併(merge)。

    Returns
    -------
    meta : pandas.DataFrame   中繼資料（含 sample_id 或其他非光譜欄位）
    X    : numpy.ndarray, shape (n_spectra, n_points)   光譜強度矩陣
    axis : numpy.ndarray, shape (n_points,)             x 軸，已由小到大排序
    """
    cols = list(df.columns)

    if isinstance(cols[0], str) and cols[0].startswith("x:"):
        meta = df.iloc[:, [0]].rename(columns={cols[0]: "sample_id"}).reset_index(drop=True)
        spec_cols = cols[1:]
    else:
        spec_cols = [c for c in cols if _is_number(c)]
        meta_cols = [c for c in cols if c not in spec_cols]
        meta = df[meta_cols].reset_index(drop=True)

    axis = np.array([float(c) for c in spec_cols])
    X = df[spec_cols].to_numpy(dtype=float)

    order = np.argsort(axis)
    axis = axis[order]
    X = X[:, order]
    return meta, X, axis


def plot_spectra(axis, X, labels, ax=None, title=None, xlabel="x", ylabel="Intensity (a.u.)"):
    """畫出多條光譜，依 labels 分色，圖例只顯示每個 label 一次。"""
    if ax is None:
        _, ax = plt.subplots(figsize=(9, 5))
    labels = list(labels)
    unique_labels = sorted(set(labels), key=labels.index)
    cmap = plt.get_cmap("tab10")
    color_of = {lab: cmap(i % 10) for i, lab in enumerate(unique_labels)}
    seen = set()
    for i in range(X.shape[0]):
        lab = labels[i]
        show_label = lab if lab not in seen else None
        seen.add(lab)
        ax.plot(axis, X[i], color=color_of[lab], linewidth=1.3, alpha=0.85, label=show_label)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title:
        ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    return ax

In [ ]:
# ---- Embedded demo spectra (fallback) ---------------------------------
# Exact content of data/nb13_demo_spectra.csv.
SAMPLE_CSV = """sample_id,group,instrument,rep,role,split,aspartame_pct,sucralose_pct,erythritol_pct,400,420,440,460,480,500,520,540,560,580,600,620,640,660,680,700,720,740,760,780,800,820,840,860,880,900,920,940,960,980,1000,1020,1040,1060,1080,1100,1120,1140,1160,1180,1200,1220,1240,1260,1280,1300,1320,1340,1360,1380,1400,1420,1440,1460,1480,1500,1520,1540,1560,1580,1600,1620,1640,1660,1680,1700,1720,1740,1760,1780,1800,class
CAL-00,0,sim_Raman785_demo,1,mixture,calibration,0.0,0.0,100.0,0.13241716,0.13081896,0.13649182,0.12414104,0.13154191,0.12723535,0.12998246,0.13582088,0.1321283,0.14478895,0.13683034,0.13187759,0.12620275,0.1287561,0.13289839,0.13146652,0.12931346,0.13218902,0.13381668,0.12273686,0.12911149,0.16047615,0.30513741,0.60500367,0.76882254,0.59173515,0.30158638,0.16251104,0.13147477,0.16222213,0.27243884,0.56807794,0.87511603,0.87743367,0.57920764,0.27909348,0.1584319,0.13267994,0.12917493,0.12514182,0.14178638,0.16267876,0.26051005,0.42862374,0.57760329,0.58708269,0.42285151,0.25945866,0.1626314,0.14128037,0.13561596,0.13642755,0.13480679,0.12121789,0.13337232,0.13627707,0.13932284,0.12348035,0.14067507,0.1394449,0.12710446,0.13412282,0.1205012,0.13692221,0.11976236,0.14102479,0.13568102,0.13416042,0.12854021,0.12455269,0.12665742,
CAL-00,0,sim_Raman785_demo,2,mixture,calibration,0.0,0.0,100.0,0.12149945,0.11316828,0.12984665,0.13078777,0.13133462,0.12968232,0.13007838,0.1293043,0.12166859,0.12913967,0.13552953,0.1218833,0.12957046,0.12342673,0.12984111,0.1211904,0.13084392,0.12544607,0.11756318,0.127897,0.12712527,0.15190558,0.31591334,0.5975362,0.77041714,0.59422322,0.29384699,0.15410696,0.14184466,0.14977316,0.2670655,0.56850372,0.87331836,0.88125355,0.56857823,0.27333845,0.16014454,0.13025447,0.1255314,0.11742347,0.1311839,0.16317418,0.25557724,0.42678145,0.58084256,0.57986905,0.42531647,0.24367613,0.15642725,0.12787353,0.12425777,0.1403843,0.12250224,0.12275131,0.1273592,0.12095692,0.1291973,0.11878508,0.13534077,0.13013314,0.12206886,0.12300404,0.12099472,0.12233967,0.12018555,0.13727488,0.12276367,0.12441971,0.1237849,0.12264089,0.1253972,
CAL-01,0,sim_Raman785_demo,1,mixture,calibration,0.0,2.5,97.5,0.18796112,0.1870373,0.17277074,0.17963748,0.17442628,0.17156658,0.17368068,0.17224905,0.18033186,0.1653873,0.18358105,0.18742569,0.20715278,0.1933525,0.18118471,0.17672475,0.17360308,0.17657434,0.19743804,0.19407367,0.1840347,0.20307235,0.34571141,0.63540892,0.79784456,0.64104401,0.35405365,0.21851804,0.18186762,0.20192939,0.3164476,0.61332738,0.90665647,0.91410764,0.61841758,0.31981697,0.19593345,0.17598955,0.1736966,0.17083205,0.18597812,0.20808876,0.30462905,0.46967095,0.62009609,0.61168813,0.46738097,0.29818893,0.21151788,0.18341259,0.16658388,0.18733428,0.17994353,0.17921363,0.16823206,0.18089018,0.16498313,0.17291575,0.17509613,0.18169787,0.17317614,0.17631821,0.18002835,0.17886615,0.17815941,0.17611926,0.18065165,0.17801673,0.18117999,0.17251764,0.17957902,
CAL-01,0,sim_Raman785_demo,2,mixture,calibration,0.0,2.5,97.5,0.1649407,0.15519179,0.15568153,0.14838546,0.14405079,0.14721143,0.15024721,0.15495363,0.15485597,0.15262139,0.14499648,0.16025228,0.1779777,0.16550556,0.15699767,0.15462816,0.1497669,0.14900357,0.15882328,0.17586632,0.16285583,0.1874161,0.33158913,0.61963244,0.77273344,0.60778365,0.31327094,0.1847389,0.15607972,0.17114214,0.29577027,0.58333154,0.87830565,0.87890684,0.58308589,0.30225689,0.18369697,0.1556308,0.1482054,0.15350175,0.15255971,0.1780906,0.27688744,0.43733447,0.59134304,0.58662545,0.44150224,0.27056545,0.18034184,0.15526058,0.1446845,0.15273225,0.13524603,0.15202358,0.15708685,0.15824172,0.15388619,0.14249395,0.15275109,0.14515606,0.1465123,0.152769,0.14867813,0.15487398,0.14903498,0.15421619,0.14686376,0.15141512,0.16300004,0.15602842,0.15669538,
CAL-02,0,sim_Raman785_demo,1,mixture,calibration,0.0,5.0,95.0,0.0920156,0.08998844,0.08925579,0.08179526,0.07984013,0.09374899,0.07292684,0.07500968,0.08755161,0.08217004,0.08694695,0.09863312,0.13641245,0.12988883,0.08953619,0.08592557,0.08663788,0.08063092,0.10029118,0.1291833,0.10166831,0.10780602,0.26263203,0.55072972,0.69432805,0.53496418,0.2526283,0.1208358,0.08487804,0.10877777,0.22505082,0.51165093,0.79433039,0.80486763,0.5399538,0.22979817,0.11580619,0.09215285,0.09020416,0.09388178,0.09027925,0.11956022,0.19806349,0.37584068,0.50593787,0.50666194,0.36816597,0.20357194,0.12647038,0.09147278,0.09007011,0.09479259,0.09256249,0.08588639,0.08242955,0.08761567,0.08674487,0.09300636,0.08866155,0.09151835,0.07757012,0.07813412,0.08974434,0.08257823,0.08410235,0.08766153,0.08024597,0.08508176,0.07586691,0.0953834,0.07936989,
CAL-02,0,sim_Raman785_demo,2,mixture,calibration,0.0,5.0,95.0,0.06767627,0.0693525,0.05500787,0.07099309,0.07572298,0.0645898,0.058785,0.06870107,0.07069215,0.07013941,0.07308718,0.08439616,0.12219554,0.11488424,0.0809814,0.05870641,0.05944877,0.06944496,0.08833317,0.10941914,0.08437666,0.09223769,0.23858635,0.51740904,0.6891378,0.51755145,0.23210278,0.09814133,0.07990843,0.0872571,0.19966026,0.49321036,0.78185671,0.7848917,0.51088658,0.2046605,0.09753925,0.07701298,0.07598613,0.06835106,0.0688383,0.10419181,0.1889028,0.34937888,0.50020733,0.49755425,0.3501569,0.18477432,0.09724146,0.07202864,0.06282611,0.06477524,0.05605969,0.06358057,0.06368594,0.05768959,0.07271746,0.06245057,0.06644655,0.06647315,0.06511141,0.06910589,0.06139264,0.07304145,0.06849545,0.07378043,0.06518953,0.07591882,0.0738315,0.06017756,0.08010724,
CAL-03,0,sim_Raman785_demo,1,mixture,calibration,0.0,7.5,92.5,0.05714878,0.05437119,0.06376986,0.05144167,0.05932587,0.0524005,0.05639569,0.05408677,0.05261605,0.05916146,0.0513349,0.07342174,0.1378434,0.13097522,0.06581647,0.06051106,0.05029459,0.05726521,0.07828398,0.11735774,0.07931144,0.08605041,0.21743886,0.49727073,0.65592308,0.49734061,0.21554332,0.09782286,0.05933214,0.076768,0.19274214,0.46207818,0.75695383,0.76321539,0.49355975,0.21403501,0.08904817,0.05827031,0.06467465,0.05909711,0.06036645,0.0866624,0.16811525,0.32634176,0.47371822,0.47332715,0.33714769,0.1703606,0.10145443,0.05583515,0.05949256,0.0616066,0.0479889,0.05582599,0.05998211,0.04679207,0.05934269,0.04756564,0.04318327,0.05522897,0.04684009,0.06802631,0.05713038,0.0526464,0.05816559,0.04870431,0.0554015,0.05760946,0.05631921,0.05522003,0.05078514,
CAL-03,0,sim_Raman785_demo,2,mixture,calibration,0.0,7.5,92.5,0.07804864,0.06451868,0.06777625,0.07338326,0.06709936,0.08031589,0.07139603,0.07534063,0.07283741,0.07943638,0.07473791,0.09610438,0.1650562,0.14492572,0.10588633,0.08087239,0.07668547,0.07004807,0.10930461,0.13661853,0.10685454,0.10820808,0.23362668,0.50247731,0.66929907,0.51025652,0.23147018,0.102144,0.08057216,0.09413305,0.20732259,0.4892958,0.76980753,0.78625498,0.51587558,0.23967597,0.10508162,0.08219225,0.07473433,0.06964811,0.07140625,0.10647475,0.17995227,0.34923762,0.4911721,0.49794862,0.35708328,0.19647916,0.11530923,0.06870823,0.08111958,0.06554,0.07547196,0.08220271,0.08108126,0.07386789,0.07074739,0.07510859,0.07335576,0.06889596,0.08481468,0.07567368,0.07694742,0.08441209,0.07908481,0.064239,0.06268884,0.06903331,0.06626935,0.06881907,0.07555214,
CAL-04,0,sim_Raman785_demo,1,mixture,calibration,0.0,10.0,90.0,-0.07139255,-0.07008252,-0.07932983,-0.07330566,-0.06462922,-0.07925593,-0.06762126,-0.07021584,-0.06747066,-0.07082964,-0.05929451,-0.05100715,0.02105236,0.03037596,-0.03855967,-0.07865641,-0.06917429,-0.06728224,-0.02799624,0.02333193,-0.02989403,-0.04611215,0.08103136,0.3644357,0.50429741,0.35539789,0.07540259,-0.03377166,-0.05968068,-0.04198504,0.06240794,0.33633414,0.62343884,0.61652668,0.36039644,0.08529588,-0.0444135,-0.06775692,-0.06897593,-0.08289961,-0.06860351,-0.04896384,0.03903277,0.19088393,0.3324934,0.35020172,0.20347143,0.04404043,-0.04359441,-0.06999261,-0.07106259,-0.05987461,-0.08267289,-0.06854335,-0.06132677,-0.06398543,-0.07070149,-0.0734949,-0.07020967,-0.08117974,-0.07498579,-0.07246581,-0.0690495,-0.06435716,-0.06531285,-0.07016864,-0.0707991,-0.06295925,-0.07230408,-0.07201828,-0.07047884,
CAL-04,0,sim_Raman785_demo,2,mixture,calibration,0.0,10.0,90.0,-0.03038466,-0.02569974,-0.03817091,-0.03647166,-0.04416505,-0.03957455,-0.04034007,-0.03837103,-0.0274693,-0.02264268,-0.03048987,-0.00610611,0.0593125,0.06506901,0.00329284,-0.03267245,-0.03425071,-0.03752381,0.01156578,0.05579176,0.01003598,0.00386829,0.11672771,0.40334448,0.56054572,0.39625768,0.11678838,-0.00124899,-0.03291516,-0.01437232,0.10268931,0.36285487,0.66881529,0.66945312,0.39938386,0.12314932,-0.00302043,-0.02939267,-0.04729793,-0.04284426,-0.03894268,-0.0036111,0.07776551,0.24441769,0.38174469,0.38986026,0.23231603,0.07079389,0.00455861,-0.02246434,-0.0264443,-0.03001042,-0.03842457,-0.03896284,-0.02778505,-0.03813315,-0.04009074,-0.02544584,-0.04276485,-0.04027725,-0.02257338,-0.03078543,-0.04080401,-0.03448918,-0.03049791,-0.02821525,-0.03430489,-0.02659187,-0.04317925,-0.03263135,-0.04161295,
CAL-05,0,sim_Raman785_demo,1,mixture,calibration,2.5,0.0,97.5,0.15891151,0.17114361,0.16171044,0.16729854,0.1715923,0.17212438,0.17541899,0.163918,0.18466687,0.16660174,0.16750237,0.16603761,0.16225321,0.16750069,0.17242032,0.16889858,0.1644304,0.16850235,0.16417411,0.17014598,0.16718298,0.19687541,0.33569583,0.63066287,0.79946518,0.62901048,0.33288393,0.19738506,0.1754096,0.19315999,0.32575371,0.59222197,0.89220604,0.88628697,0.59248888,0.31563244,0.19350695,0.16698627,0.17171313,0.16501736,0.17081475,0.19469387,0.28057621,0.45759241,0.61482863,0.60298645,0.45275357,0.28489439,0.19622669,0.17549918,0.15896763,0.16002598,0.16318558,0.16840401,0.16599243,0.17366032,0.16551744,0.16471142,0.153084,0.16733359,0.16856624,0.18515201,0.16965972,0.15936536,0.16205255,0.16886284,0.15935341,0.16717894,0.16907633,0.16441868,0.16615479,
CAL-05,0,sim_Raman785_demo,2,mixture,calibration,2.5,0.0,97.5,0.2008776,0.20355318,0.20367766,0.19930233,0.20556866,0.19747425,0.19909156,0.200762,0.20700716,0.20690032,0.2061743,0.19519066,0.2094122,0.19624094,0.20241728,0.1950254,0.19209445,0.20638738,0.20654208,0.21088763,0.20967798,0.23189043,0.371637,0.66866545,0.82514081,0.66199552,0.37143989,0.22428868,0.2134011,0.22809704,0.3567809,0.63389292,0.92676694,0.93161157,0.6329496,0.33864878,0.22992503,0.20545339,0.21415724,0.20661431,0.2136683,0.23234167,0.32900319,0.48695561,0.63549582,0.64168598,0.48682479,0.3350277,0.24186872,0.20470867,0.20618209,0.19341973,0.20145919,0.19929044,0.19613215,0.19709733,0.19284575,0.19780381,0.2010726,0.20374864,0.20317056,0.20182784,0.20643638,0.20827895,0.19770793,0.19176051,0.19713095,0.20304869,0.21115638,0.18383193,0.20045468,
CAL-06,0,sim_Raman785_demo,1,mixture,calibration,2.5,2.5,95.0,0.24187469,0.24490224,0.23658473,0.25187037,0.2430542,0.25581148,0.24543362,0.24662549,0.25262148,0.24881174,0.25562055,0.26443255,0.27208122,0.26491252,0.2543252,0.24388213,0.24483263,0.24807518,0.26042154,0.27681882,0.26379598,0.27856688,0.4146658,0.69009823,0.85968496,0.70341592,0.4202207,0.28947764,0.23838029,0.26483126,0.40212921,0.69040055,0.95578207,0.95700083,0.67093981,0.39467447,0.28463755,0.25741724,0.24856905,0.25960446,0.2537815,0.28696058,0.36910406,0.53507853,0.6753621,0.6776145,0.53753043,0.36218563,0.28895638,0.25655776,0.24415563,0.25790276,0.25679167,0.24677462,0.2529847,0.25060958,0.24696656,0.23588588,0.24645518,0.25108982,0.25695527,0.24047551,0.24597049,0.24416873,0.25209552,0.24458207,0.25376738,0.24375179,0.24676488,0.24557118,0.24874586,
CAL-06,0,sim_Raman785_demo,2,mixture,calibration,2.5,2.5,95.0,0.25938717,0.28014971,0.26189038,0.2595113,0.25433898,0.26387827,0.26475892,0.26923628,0.25844139,0.26945236,0.2647352,0.26563629,0.28275689,0.29403247,0.27000864,0.26316503,0.26946883,0.25721974,0.28458456,0.28645682,0.27980122,0.30193941,0.42631751,0.70241117,0.88883788,0.70962674,0.43369843,0.28920988,0.26199543,0.29312728,0.41694115,0.6979284,0.97936263,0.9950448,0.69629178,0.40958576,0.29240425,0.2675501,0.25271332,0.26846604,0.26278789,0.28922421,0.38186646,0.54475647,0.68957476,0.69201473,0.55699277,0.39095578,0.2998937,0.27712439,0.25338026,0.25886981,0.25638776,0.25485317,0.27508875,0.28127833,0.27091868,0.2513596,0.25686678,0.27176974,0.27257971,0.27764452,0.26677699,0.26130292,0.26165485,0.26072059,0.26865816,0.26245312,0.27089946,0.25193987,0.26090819,
CAL-07,0,sim_Raman785_demo,1,mixture,calibration,2.5,5.0,92.5,0.118502,0.11667076,0.1115411,0.11680253,0.11405509,0.12088588,0.117624,0.11476687,0.11369138,0.11976673,0.11612227,0.12779637,0.16987603,0.16547212,0.12861387,0.11422826,0.12105517,0.11145757,0.15299382,0.1698272,0.13825192,0.15641843,0.28034551,0.55143552,0.7158878,0.56654094,0.28507585,0.15048017,0.12708791,0.13197272,0.27608054,0.5385059,0.82676427,0.82965308,0.5518401,0.24983845,0.1384072,0.12385665,0.11878977,0.11519734,0.11576028,0.14918071,0.22198065,0.40082142,0.52889218,0.54326999,0.40208911,0.24028666,0.1484458,0.12835692,0.1086918,0.11422761,0.11092401,0.11852805,0.11556946,0.12164835,0.11849095,0.12372586,0.11671523,0.11791266,0.12442,0.12015059,0.11340579,0.11831765,0.12798803,0.12024484,0.10642719,0.115994,0.11602005,0.11785569,0.11446348,
CAL-07,0,sim_Raman785_demo,2,mixture,calibration,2.5,5.0,92.5,0.09278693,0.0999362,0.09139671,0.09450035,0.09133594,0.09735016,0.0932528,0.0952372,0.09683441,0.10136042,0.08207304,0.10674068,0.13061539,0.14038402,0.10581548,0.09888837,0.10518294,0.10139199,0.11538225,0.13975857,0.12320126,0.13112186,0.26297791,0.5398705,0.70230304,0.53095709,0.26771068,0.12293697,0.10250654,0.12115699,0.25571754,0.51732671,0.81563964,0.80780523,0.53862281,0.24904367,0.13926471,0.09562783,0.09926071,0.09555592,0.10058427,0.12859659,0.23265216,0.3791497,0.51986899,0.50942194,0.37060601,0.22300232,0.13907425,0.10702259,0.09902164,0.1049234,0.10521331,0.08726003,0.09581391,0.09796015,0.10179491,0.08328122,0.09529954,0.08628754,0.10818845,0.10117867,0.11167985,0.09166822,0.10523824,0.09532869,0.09700235,0.09681274,0.10302361,0.09726635,0.10512276,
CAL-08,0,sim_Raman785_demo,1,mixture,calibration,2.5,7.5,90.0,0.11990953,0.12145803,0.12261349,0.11704244,0.12051238,0.11643584,0.11407015,0.12450354,0.12082448,0.12576141,0.1101563,0.14184938,0.19801414,0.18877015,0.14085718,0.13093126,0.13335757,0.12770343,0.14179972,0.19500645,0.16021505,0.1525212,0.27945656,0.53949511,0.69286667,0.55307921,0.27411564,0.15587817,0.12376734,0.14283394,0.27778871,0.52652418,0.81702826,0.81742235,0.54911791,0.26478209,0.15622952,0.12656765,0.12348217,0.12137239,0.12625697,0.16216031,0.23376896,0.39366838,0.53347864,0.53591677,0.38294233,0.24006801,0.15150485,0.12635156,0.12179284,0.11967504,0.13032484,0.11298122,0.11537865,0.11512769,0.12788229,0.12291979,0.12191999,0.12154789,0.12334527,0.13396818,0.12345473,0.11955749,0.12899572,0.11747678,0.12709357,0.12343999,0.12878306,0.12466513,0.12948296,
CAL-08,0,sim_Raman785_demo,2,mixture,calibration,2.5,7.5,90.0,0.14883336,0.14466687,0.15600084,0.14754334,0.14260066,0.16339395,0.14612179,0.16334259,0.16013242,0.15510245,0.15062063,0.17847211,0.21960447,0.22452073,0.15937703,0.15954162,0.16213894,0.16049,0.19504952,0.22191064,0.18950782,0.1946826,0.31092062,0.57152631,0.72167837,0.58634088,0.31243662,0.18825012,0.16207482,0.17744603,0.31512869,0.55881331,0.84689669,0.8487924,0.59496812,0.3054758,0.18398283,0.16601793,0.15175164,0.15364286,0.1574507,0.18733792,0.27409388,0.42834949,0.56239953,0.56545112,0.4256803,0.2640007,0.18437837,0.15692196,0.1589669,0.15389195,0.1593091,0.14317978,0.16712501,0.15159979,0.15083095,0.16937755,0.15888447,0.15461805,0.15376193,0.15917195,0.14526457,0.16210376,0.14644311,0.16196804,0.15162571,0.15617916,0.16232702,0.15656211,0.15025335,
CAL-09,0,sim_Raman785_demo,1,mixture,calibration,2.5,10.0,87.5,0.09058125,0.09672802,0.10810386,0.10180708,0.10076268,0.09679427,0.09927391,0.09464647,0.11583075,0.09720824,0.10462176,0.11836978,0.18446067,0.18731399,0.13666276,0.10289564,0.08886975,0.09812165,0.13416251,0.1895267,0.14520364,0.12805546,0.25584322,0.5239484,0.67022811,0.51813261,0.26531766,0.12010224,0.10124741,0.11188932,0.24941939,0.49752732,0.78362655,0.77629247,0.52235746,0.24018274,0.1312454,0.09608877,0.0897013,0.10484783,0.10605847,0.13242905,0.20325936,0.36179619,0.49546556,0.51025284,0.36224147,0.21371038,0.12694125,0.09504065,0.09696159,0.10108571,0.09204778,0.09747144,0.09745917,0.10287003,0.09212704,0.09565136,0.10513152,0.09333591,0.10932454,0.10085109,0.09580774,0.10013285,0.09690201,0.09572109,0.09681499,0.10806269,0.09625524,0.11460729,0.09946957,
CAL-09,0,sim_Raman785_demo,2,mixture,calibration,2.5,10.0,87.5,0.16104181,0.16386596,0.16498245,0.15324026,0.15705708,0.15973302,0.15257149,0.14952004,0.16796361,0.14758025,0.15329545,0.17822247,0.25630449,0.25591855,0.17880324,0.15780896,0.15910197,0.16370308,0.20468027,0.25637374,0.21068629,0.18401106,0.31087107,0.57936693,0.72806425,0.57788682,0.3168099,0.18260575,0.16452501,0.18858684,0.31110725,0.55429108,0.814084,0.84352136,0.59555958,0.30597673,0.17219696,0.15641057,0.15147375,0.16118767,0.15652778,0.19219554,0.27409817,0.41766788,0.56058362,0.55432756,0.41950927,0.27256662,0.19149192,0.15644736,0.14772403,0.15406007,0.15958412,0.15694128,0.15316424,0.16778909,0.16126932,0.14533706,0.14878799,0.15567644,0.16265284,0.15219994,0.15223297,0.161628,0.15859832,0.15187731,0.16083893,0.14727528,0.16391478,0.16294174,0.16630238,
CAL-10,0,sim_Raman785_demo,1,mixture,calibration,5.0,0.0,95.0,0.25749626,0.26024186,0.2617188,0.26526969,0.26305375,0.25855231,0.25965216,0.26502589,0.2631258,0.26206402,0.25942711,0.25632588,0.24883456,0.25076545,0.25725781,0.24886168,0.25385156,0.26048152,0.25656487,0.27201735,0.26505407,0.29073818,0.42595841,0.70621323,0.85972734,0.7156918,0.41910598,0.2926104,0.27474643,0.29057479,0.43959956,0.69193595,0.9762667,0.97478826,0.68023735,0.39896243,0.28482436,0.26917981,0.25960984,0.26664214,0.279561,0.29054486,0.37979069,0.54550331,0.68963296,0.68714341,0.53956646,0.37847568,0.28978091,0.2609792,0.26283309,0.26130992,0.26192609,0.26208827,0.25321303,0.25878652,0.26269603,0.25557958,0.25957608,0.25664987,0.27188726,0.27709721,0.2551184,0.24892572,0.26472385,0.25690346,0.25036657,0.2618343,0.26127604,0.2637789,0.24840062,
CAL-10,0,sim_Raman785_demo,2,mixture,calibration,5.0,0.0,95.0,0.28685472,0.28765003,0.28784075,0.28155183,0.27848703,0.28164329,0.27981282,0.28228942,0.27738885,0.27959453,0.27552596,0.28846667,0.28724566,0.27928887,0.29183971,0.28137472,0.28756366,0.28250236,0.28269483,0.29095019,0.29272028,0.31837737,0.44860753,0.7405945,0.89247662,0.73914954,0.45109935,0.30101764,0.28780122,0.31869259,0.46314801,0.71193313,1.00297773,0.99215797,0.69845945,0.41455657,0.30961548,0.29128156,0.28329099,0.29708249,0.29046152,0.31391858,0.40578637,0.5671696,0.71968903,0.71782414,0.57324322,0.39662967,0.31994846,0.28526626,0.26654301,0.28074061,0.27655391,0.2816624,0.29137936,0.27869308,0.29023135,0.28577281,0.28433349,0.28431867,0.30435469,0.29180526,0.28375429,0.28069362,0.28117644,0.27612969,0.28783478,0.27989445,0.28837145,0.28321301,0.27843555,
CAL-11,0,sim_Raman785_demo,1,mixture,calibration,5.0,2.5,92.5,0.15332456,0.15144738,0.14718488,0.14361545,0.14864446,0.14356739,0.14927968,0.15149255,0.14844921,0.14003002,0.14969448,0.14676933,0.16620422,0.16775876,0.15657032,0.14636471,0.14945588,0.16246762,0.15259121,0.17753093,0.15292679,0.17172712,0.30679431,0.58523822,0.74777377,0.58429393,0.30061856,0.16925056,0.1540927,0.17121111,0.32715747,0.56643403,0.84969343,0.85912653,0.56654965,0.28921831,0.1760696,0.14191945,0.13360275,0.13607784,0.15875599,0.17859297,0.26631409,0.41284729,0.5646487,0.56386669,0.43240624,0.26490153,0.19311813,0.15953623,0.14880813,0.15901761,0.15754088,0.146668,0.15372395,0.14569805,0.15222979,0.15562467,0.14612811,0.13890223,0.15157161,0.15855719,0.15242192,0.14690436,0.14794782,0.15572838,0.14445531,0.14948691,0.14514361,0.15334248,0.14062358,
CAL-11,0,sim_Raman785_demo,2,mixture,calibration,5.0,2.5,92.5,0.09885832,0.10214931,0.09877188,0.11248243,0.10710423,0.10629671,0.10183185,0.10172471,0.10753355,0.10945938,0.10260802,0.10688178,0.13512129,0.1333778,0.12114846,0.11130553,0.1109995,0.10691318,0.11229806,0.13947807,0.12669814,0.12351449,0.26160517,0.55486726,0.71203877,0.55204404,0.26769059,0.13638761,0.11279311,0.13373656,0.29213559,0.51851914,0.8134749,0.81628996,0.53517305,0.23955614,0.1377571,0.11460541,0.10373773,0.11607131,0.11495518,0.13027585,0.22455608,0.38358845,0.51584969,0.52382609,0.39265006,0.22107753,0.14410122,0.10819906,0.10841469,0.10590267,0.10676232,0.11500946,0.10878489,0.10720041,0.10831143,0.11572188,0.10742394,0.10246404,0.12333543,0.12342294,0.10927143,0.10965116,0.11384468,0.11973629,0.10891677,0.10634961,0.11461113,0.10698745,0.09858251,
CAL-12,0,sim_Raman785_demo,1,mixture,calibration,5.0,5.0,90.0,0.09918219,0.09097771,0.09170904,0.08797095,0.08934852,0.08533063,0.09233921,0.09021406,0.08124082,0.08886487,0.08664289,0.10086467,0.13991876,0.12849576,0.10290282,0.09109123,0.09142053,0.08222186,0.11213735,0.14161644,0.11321768,0.11962774,0.24471795,0.51821236,0.67551313,0.5181329,0.24831451,0.12244844,0.09143526,0.12163785,0.26523187,0.49724869,0.7812634,0.78080551,0.50240303,0.2363225,0.11542834,0.09200568,0.09488253,0.09971157,0.09137898,0.11399053,0.21389155,0.3693447,0.50367008,0.49774699,0.3699143,0.21303274,0.12653736,0.09487921,0.08971762,0.09118218,0.09526132,0.10242468,0.08990325,0.08835179,0.08506142,0.10019322,0.09643139,0.09985871,0.10051329,0.10260402,0.0982284,0.09079223,0.09351388,0.07972451,0.09619329,0.09176891,0.10024676,0.08150915,0.09073402,
CAL-12,0,sim_Raman785_demo,2,mixture,calibration,5.0,5.0,90.0,0.108837,0.11417583,0.09575558,0.09997224,0.10897842,0.11321594,0.10078869,0.09452665,0.10950327,0.1122606,0.10062997,0.1062996,0.15223877,0.15116028,0.12315661,0.1080748,0.09925225,0.10598258,0.13947193,0.15075307,0.13292893,0.13852155,0.25785185,0.52890414,0.68017784,0.52972089,0.25939429,0.13481908,0.11298861,0.12969138,0.27017665,0.52689909,0.80317723,0.79861198,0.53324128,0.24588772,0.13131926,0.10576328,0.10842611,0.09997958,0.10239757,0.13272022,0.22583823,0.37518827,0.51412794,0.50860084,0.37818727,0.22027289,0.12166282,0.11397642,0.10144574,0.1078071,0.10779841,0.10563222,0.10296172,0.09610612,0.11517047,0.11129736,0.10823049,0.10723673,0.12034849,0.10858535,0.10811612,0.10237641,0.09834464,0.10051889,0.10478373,0.09475554,0.09150798,0.10017869,0.10331545,
CAL-13,0,sim_Raman785_demo,1,mixture,calibration,5.0,7.5,87.5,0.13090541,0.13400149,0.14974678,0.14267238,0.13314071,0.14309947,0.13250896,0.14230114,0.13257173,0.13707594,0.13850134,0.15821125,0.20446949,0.21131106,0.15336057,0.14857747,0.13100286,0.14175015,0.17101645,0.20022493,0.16370687,0.17869759,0.2937334,0.54322554,0.70673015,0.54925741,0.29635259,0.16052835,0.13841094,0.16661296,0.31536035,0.54177874,0.81768165,0.81529386,0.56336769,0.27828647,0.15838723,0.14734254,0.1378711,0.13407377,0.13313472,0.16550357,0.24509246,0.41655786,0.54370407,0.53979608,0.40828065,0.26316323,0.16967938,0.14282132,0.14293105,0.13661404,0.14089531,0.13135663,0.1356144,0.13014454,0.14645907,0.13810478,0.14270588,0.13841514,0.13562516,0.13884055,0.14530197,0.13294964,0.13883648,0.13522797,0.13930893,0.1394464,0.13169586,0.14230893,0.1389081,
CAL-13,0,sim_Raman785_demo,2,mixture,calibration,5.0,7.5,87.5,0.11585549,0.11396834,0.11905012,0.12331217,0.13016611,0.1271526,0.11648692,0.11858099,0.12364247,0.12705646,0.12131409,0.14237296,0.18514346,0.19399379,0.14293292,0.11328679,0.11810735,0.12230067,0.15270775,0.19310342,0.16209569,0.1437451,0.27194625,0.53616327,0.68960656,0.54075705,0.26280546,0.14716832,0.12521181,0.15289368,0.28674425,0.52101453,0.78609221,0.80141825,0.53264439,0.26344428,0.15300068,0.12391575,0.12954575,0.1225912,0.13184251,0.15199715,0.23232286,0.3840644,0.51288993,0.52017648,0.3744222,0.23140432,0.14902771,0.12550452,0.11438019,0.12051371,0.11934042,0.12369036,0.11732396,0.11557765,0.11529716,0.10269788,0.12498137,0.12023942,0.12828969,0.12909052,0.12297314,0.11277873,0.11778244,0.11782039,0.12237443,0.11739815,0.11636019,0.12398389,0.11260908,
CAL-14,0,sim_Raman785_demo,1,mixture,calibration,5.0,10.0,85.0,0.12934214,0.12649556,0.12215876,0.15017085,0.13445424,0.13318522,0.13576184,0.13052816,0.13108482,0.1261726,0.13897109,0.16654285,0.23329871,0.23105025,0.16599245,0.14567723,0.13131364,0.13641696,0.17415448,0.23569561,0.18731349,0.16794552,0.27915251,0.53864404,0.69663128,0.54401169,0.29105222,0.15959425,0.13956634,0.16163426,0.29790229,0.5277975,0.7945983,0.81619278,0.5575756,0.28842531,0.15553173,0.12937335,0.13079998,0.14076691,0.14839004,0.17426265,0.24506442,0.37814423,0.53044312,0.52299178,0.38723233,0.247524,0.15805697,0.13058565,0.13277019,0.14052224,0.12275752,0.1296985,0.14157289,0.13157582,0.12545013,0.13114996,0.13788295,0.14664097,0.15516613,0.14760546,0.1352238,0.14139172,0.14572575,0.13670763,0.14013555,0.13543217,0.1431846,0.1306926,0.1438487,
CAL-14,0,sim_Raman785_demo,2,mixture,calibration,5.0,10.0,85.0,0.15563428,0.14140186,0.16074486,0.1650398,0.14798834,0.16245058,0.15298246,0.1592371,0.14604272,0.15949541,0.15718257,0.18039062,0.24224493,0.24608006,0.17631785,0.16087581,0.14998318,0.15555103,0.19504305,0.23899408,0.19957031,0.18975038,0.2927827,0.55628013,0.71874629,0.55647139,0.29929308,0.1727287,0.16015076,0.17655915,0.32041761,0.54297975,0.81016158,0.8315304,0.57339406,0.31115582,0.18389362,0.14896062,0.16500601,0.14251619,0.15700656,0.18475643,0.25663512,0.40649045,0.54004664,0.54790427,0.41499089,0.26753185,0.17886542,0.15995876,0.15895442,0.15712052,0.14310526,0.1546345,0.15424881,0.15530979,0.15026031,0.15497076,0.14856877,0.15051148,0.16543085,0.15402004,0.15441595,0.15675005,0.1476964,0.14782792,0.14406698,0.15846941,0.15720006,0.15067678,0.14837312,
CAL-15,0,sim_Raman785_demo,1,mixture,calibration,7.5,0.0,92.5,0.12484826,0.11618137,0.11828931,0.1157598,0.11907164,0.11906178,0.12378559,0.12585018,0.1215187,0.12199465,0.11534587,0.12782442,0.1291943,0.11308994,0.12883917,0.12174807,0.1185832,0.12205873,0.12516055,0.13749973,0.13134429,0.14787953,0.28612465,0.55438617,0.71759197,0.5633526,0.28120045,0.14596417,0.11967013,0.14142118,0.33123362,0.54177423,0.8113346,0.82033941,0.55163089,0.26209131,0.14197805,0.12029935,0.1207771,0.12304376,0.13677778,0.15651178,0.23298145,0.4090114,0.54772391,0.54210885,0.40892393,0.246803,0.15253014,0.12258074,0.11816578,0.12333431,0.1250512,0.1228945,0.12155826,0.11349537,0.1184884,0.12116972,0.11073234,0.12343695,0.13194185,0.13812071,0.13467709,0.11212328,0.12370542,0.12608533,0.12152806,0.12482871,0.12222547,0.11436558,0.12075679,
CAL-15,0,sim_Raman785_demo,2,mixture,calibration,7.5,0.0,92.5,0.12768357,0.13085713,0.13742129,0.12939025,0.14355297,0.13670982,0.13369113,0.13220424,0.13512281,0.13046827,0.13288249,0.13960249,0.13180996,0.1390313,0.13243187,0.12755636,0.12743792,0.12770928,0.13585369,0.14745769,0.13843863,0.16461248,0.2965044,0.56802051,0.73070168,0.56610373,0.28892523,0.17131024,0.13437125,0.15879702,0.33186759,0.55345191,0.8354633,0.82832305,0.54469911,0.26813552,0.15260075,0.11858355,0.13400189,0.1329123,0.13608118,0.16607703,0.23533744,0.40231401,0.5421477,0.5515575,0.41812567,0.2665394,0.17348215,0.14392855,0.13504534,0.13475583,0.13743314,0.13753479,0.13414437,0.12748295,0.13380469,0.1321225,0.13189342,0.14402472,0.15109415,0.14426106,0.14460018,0.13729851,0.12274289,0.13342653,0.13718176,0.12612299,0.12512327,0.13379455,0.14021524,
CAL-16,0,sim_Raman785_demo,1,mixture,calibration,7.5,2.5,90.0,0.14939743,0.14665166,0.14330741,0.14224901,0.13715991,0.15535398,0.15552096,0.13995921,0.15638823,0.14654646,0.14856862,0.14755608,0.18203341,0.17547902,0.14973461,0.1465499,0.14255675,0.15115398,0.17490637,0.17963551,0.17253189,0.17847271,0.31197322,0.58535195,0.73105228,0.58443106,0.31137211,0.1808718,0.15654221,0.16989237,0.33898259,0.55937099,0.84382131,0.83125525,0.56352972,0.28373206,0.16731625,0.13778449,0.15273111,0.14234359,0.15245171,0.1795392,0.26373033,0.41543035,0.55817206,0.56645419,0.41517035,0.27039021,0.18600574,0.14822488,0.15109505,0.14392108,0.14789899,0.1473421,0.1488769,0.15407678,0.13507216,0.13892706,0.15130306,0.15367578,0.16737762,0.16949766,0.15334556,0.14678544,0.15594527,0.15804452,0.13813489,0.14472704,0.14889312,0.13760221,0.14765919,
CAL-16,0,sim_Raman785_demo,2,mixture,calibration,7.5,2.5,90.0,0.13516346,0.1482555,0.13482145,0.14031759,0.1380797,0.13949964,0.1437571,0.12743424,0.14410924,0.13196933,0.14758358,0.135578,0.16692684,0.16438407,0.14257148,0.14130611,0.13666328,0.13234283,0.15766529,0.17237635,0.153237,0.16246082,0.29660633,0.55946238,0.72929327,0.56136726,0.29825188,0.16677627,0.14578246,0.15822629,0.32150491,0.54530784,0.81213203,0.82133485,0.53758654,0.27783357,0.16995234,0.12905958,0.13151777,0.1371993,0.14347114,0.17168063,0.24198371,0.41479218,0.5480044,0.54162531,0.41166115,0.26717761,0.15441183,0.13659806,0.13954107,0.14205074,0.13847984,0.13818309,0.14046199,0.12190036,0.13533936,0.13957045,0.13479794,0.14094405,0.15809107,0.15153393,0.14272736,0.13347466,0.13275952,0.14534764,0.13227709,0.1252441,0.12704556,0.13231938,0.13766503,
CAL-17,0,sim_Raman785_demo,1,mixture,calibration,7.5,5.0,87.5,-0.10238925,-0.10429618,-0.10263299,-0.10390238,-0.10853063,-0.10193525,-0.10033814,-0.09793897,-0.11344474,-0.11669607,-0.10873328,-0.09038658,-0.05624673,-0.0626755,-0.07851672,-0.10250793,-0.10326509,-0.10256229,-0.09376971,-0.058763,-0.08232578,-0.06186355,0.05097084,0.31744602,0.47144206,0.315109,0.06039362,-0.08100491,-0.10293916,-0.07976304,0.09140381,0.29225517,0.56825373,0.58060801,0.29214313,0.04355172,-0.07596238,-0.1106136,-0.09116094,-0.11208946,-0.0917224,-0.08185228,0.00396047,0.16270624,0.3045382,0.28811879,0.16747156,0.01961272,-0.06981573,-0.09283591,-0.10331091,-0.10089985,-0.09504266,-0.10587329,-0.1089863,-0.10730347,-0.10168014,-0.10376016,-0.10870551,-0.09048637,-0.08776119,-0.08559177,-0.09521116,-0.11208829,-0.10324079,-0.11099042,-0.10403685,-0.10466193,-0.11649323,-0.11132496,-0.11580085,
CAL-17,0,sim_Raman785_demo,2,mixture,calibration,7.5,5.0,87.5,-0.07636818,-0.08038881,-0.07775805,-0.08712321,-0.08299562,-0.07528499,-0.08070981,-0.0711731,-0.07844066,-0.07927321,-0.08019485,-0.07086015,-0.02711154,-0.03360576,-0.05720995,-0.07930409,-0.07655919,-0.07475219,-0.06909363,-0.03318253,-0.0478145,-0.04858969,0.07220559,0.32962795,0.495242,0.33722548,0.07302422,-0.04992621,-0.07846753,-0.0526035,0.11965561,0.32577301,0.59345447,0.59603436,0.33300808,0.06414138,-0.0526267,-0.09072301,-0.06604304,-0.08397797,-0.084229,-0.04319461,0.03870862,0.18300528,0.31004917,0.32275942,0.19858507,0.04694013,-0.04191992,-0.073051,-0.07973489,-0.08012667,-0.08572958,-0.07598363,-0.08631354,-0.08804345,-0.07398564,-0.07776397,-0.07618128,-0.07306408,-0.05310795,-0.06568543,-0.08773112,-0.08129634,-0.08772252,-0.0801283,-0.07534528,-0.07039474,-0.0872529,-0.07860637,-0.07762854,
CAL-18,0,sim_Raman785_demo,1,mixture,calibration,7.5,7.5,85.0,0.12691521,0.11798057,0.13976639,0.13850491,0.1251441,0.13239539,0.12261694,0.13494144,0.14285228,0.13074571,0.12719162,0.14536845,0.19322172,0.2022506,0.15299504,0.14279505,0.12433883,0.13599156,0.15382901,0.21311961,0.15929139,0.16814469,0.29224916,0.53065664,0.70093531,0.53607004,0.28611133,0.16449678,0.14021655,0.15192588,0.32387486,0.51701234,0.79661822,0.80269935,0.5498734,0.28021694,0.15812772,0.13201435,0.13759299,0.12310146,0.14543355,0.16179452,0.23906599,0.38668437,0.52331661,0.52077326,0.38834658,0.24113136,0.16054701,0.14041897,0.1250065,0.1274393,0.13391318,0.12760192,0.13202548,0.1295368,0.13312571,0.1261172,0.12451863,0.12719828,0.14386212,0.1419029,0.13478979,0.12930729,0.13177446,0.13519733,0.12807596,0.13173265,0.14168898,0.13792384,0.13630998,
CAL-18,0,sim_Raman785_demo,2,mixture,calibration,7.5,7.5,85.0,0.16273673,0.16149784,0.16388031,0.16432634,0.16539016,0.17107832,0.16050032,0.16616521,0.16034736,0.15768194,0.18117568,0.18999115,0.23852308,0.24238094,0.19912992,0.18178377,0.16604735,0.16794543,0.21236098,0.24813179,0.20287071,0.20225088,0.30908722,0.56677141,0.72050236,0.56945763,0.31307582,0.19614739,0.1835127,0.18278231,0.37209255,0.56072986,0.82031038,0.84063658,0.57932106,0.30507875,0.19940595,0.1709013,0.16501895,0.16829117,0.18288974,0.20070998,0.27378924,0.41341773,0.56146419,0.54459138,0.44066936,0.2740001,0.21594189,0.17524782,0.17546379,0.16023767,0.15482867,0.16403833,0.15941658,0.16148097,0.16743598,0.16637367,0.1603239,0.1725581,0.19316548,0.19170799,0.1715199,0.16338892,0.17390162,0.15655489,0.16789232,0.16884643,0.16046131,0.17950156,0.17203931,
CAL-19,0,sim_Raman785_demo,1,mixture,calibration,7.5,10.0,82.5,0.08432925,0.06154933,0.06545421,0.06403293,0.06816392,0.07423217,0.07522538,0.07491125,0.07965911,0.08273197,0.07380343,0.09859626,0.1604211,0.16162027,0.09090092,0.08470297,0.07084037,0.08129471,0.12566006,0.18039475,0.13532579,0.11181056,0.21387444,0.46020392,0.62471927,0.47938691,0.2259265,0.10780638,0.07269848,0.09469159,0.26532387,0.45157462,0.71429379,0.72280446,0.48445366,0.22344656,0.10226483,0.08300338,0.07407643,0.07678414,0.08841148,0.10679346,0.1855725,0.32295728,0.46500539,0.45188354,0.32740471,0.18537478,0.1109435,0.08428684,0.07694184,0.06750995,0.06957884,0.06463609,0.07937256,0.06887952,0.07491347,0.07572567,0.05838013,0.07011653,0.08841567,0.08903406,0.07678643,0.08540518,0.07567307,0.0872589,0.08276993,0.07495483,0.06683274,0.0712976,0.08244876,
CAL-19,0,sim_Raman785_demo,2,mixture,calibration,7.5,10.0,82.5,0.02593474,0.02471145,0.0371862,0.03885614,0.03216339,0.03732989,0.02901589,0.02876043,0.03749003,0.04090804,0.03243743,0.06375401,0.13507473,0.12194643,0.06261437,0.03349844,0.03898577,0.03374544,0.07857271,0.12648226,0.09389187,0.07632927,0.18309408,0.43489382,0.58238022,0.44126379,0.17300511,0.07140144,0.03410385,0.05567404,0.22622012,0.41444993,0.682155,0.6928752,0.44197493,0.18045236,0.0742644,0.03424106,0.02176433,0.03362991,0.03839258,0.06177361,0.1342732,0.2897614,0.40945015,0.41957984,0.28195489,0.14880018,0.06969249,0.04105188,0.03111463,0.04621476,0.04054223,0.02940025,0.0309922,0.05344045,0.04701281,0.03149161,0.03160777,0.04404609,0.04932731,0.05999867,0.03681315,0.03435677,0.03292434,0.02772025,0.0405048,0.03187101,0.04933157,0.03732337,0.02820084,
CAL-20,0,sim_Raman785_demo,1,mixture,calibration,10.0,0.0,90.0,0.20121662,0.20353471,0.21185009,0.20026384,0.21357747,0.20270302,0.19439132,0.19205208,0.20526637,0.19674247,0.21094478,0.20124081,0.20377454,0.20123889,0.19278862,0.19252258,0.2013497,0.21000455,0.211091,0.21498755,0.2106085,0.23377568,0.35742255,0.6459493,0.77904505,0.62814837,0.36557959,0.23384225,0.20733226,0.22438468,0.41924559,0.62312333,0.8878496,0.8908854,0.59826013,0.33515684,0.21443198,0.21268269,0.20560394,0.21051884,0.21137837,0.23698624,0.31895451,0.47113079,0.60887344,0.61093573,0.48283075,0.33424204,0.23540017,0.20733915,0.20913579,0.19755104,0.19469631,0.19920702,0.19586342,0.19645289,0.211881,0.19982981,0.19909494,0.20176492,0.23348298,0.22691826,0.20292702,0.20226846,0.20333706,0.20422899,0.20887913,0.1939203,0.20687132,0.20479218,0.20664663,
CAL-20,0,sim_Raman785_demo,2,mixture,calibration,10.0,0.0,90.0,0.1880269,0.1864936,0.1783067,0.18062706,0.18732441,0.18272508,0.17869791,0.18980214,0.1812818,0.18600805,0.17651241,0.1917361,0.19603321,0.17107709,0.19060195,0.17257663,0.18879926,0.18343706,0.18440948,0.20578053,0.19513586,0.20932274,0.34420974,0.61359085,0.7676071,0.61366955,0.3484646,0.21385756,0.18897862,0.20815579,0.40880805,0.598652,0.86890638,0.8705337,0.58399385,0.31508612,0.21559362,0.18168011,0.19076392,0.18690274,0.19693969,0.21030186,0.29764436,0.44334007,0.58555261,0.59907329,0.46934215,0.30262209,0.22057468,0.19877869,0.17566131,0.17977761,0.18372306,0.18389441,0.1781295,0.18446239,0.17707597,0.18916496,0.18242033,0.19113734,0.21248208,0.21344388,0.18143051,0.18657821,0.18398102,0.17762981,0.18562431,0.17994582,0.17269643,0.18722349,0.17984482,
CAL-21,0,sim_Raman785_demo,1,mixture,calibration,10.0,2.5,87.5,-0.06799004,-0.05372321,-0.06392327,-0.07735788,-0.05545549,-0.06578147,-0.06052931,-0.06438468,-0.06495405,-0.06709392,-0.06397931,-0.04272099,-0.03781735,-0.03723628,-0.05396337,-0.06013375,-0.0490532,-0.05567446,-0.04092365,-0.0310073,-0.02932777,-0.02737151,0.08821643,0.35425849,0.51908944,0.35234497,0.09822091,-0.02700647,-0.05278803,-0.0353827,0.15783711,0.33792249,0.60750214,0.60007308,0.33966601,0.07651715,-0.03611318,-0.04820098,-0.06806851,-0.07163373,-0.05990092,-0.03085341,0.04379959,0.20346258,0.33551216,0.33817647,0.20671642,0.05730613,-0.01381642,-0.06123425,-0.06070738,-0.05530405,-0.06583295,-0.06662161,-0.06363247,-0.06573125,-0.06664724,-0.06368897,-0.05668458,-0.05471965,-0.03125702,-0.0368063,-0.06090199,-0.06432838,-0.05420408,-0.06776994,-0.06418931,-0.05450258,-0.06123995,-0.05572291,-0.06014801,
CAL-21,0,sim_Raman785_demo,2,mixture,calibration,10.0,2.5,87.5,-0.02853875,-0.04242938,-0.0313332,-0.02648654,-0.02363771,-0.02487869,-0.0269195,-0.03394851,-0.01778758,-0.03374045,-0.02051276,-0.02579559,-0.01164702,-0.00615604,-0.01842653,-0.02497962,-0.03642167,-0.03421201,-0.0194752,0.0097801,-0.00438341,-0.00111349,0.11802189,0.38772633,0.53355535,0.38645025,0.12793872,0.0008422,-0.02595978,-0.00236341,0.18064925,0.36650041,0.63847978,0.64854767,0.36134225,0.09147181,-0.00864641,-0.01939125,-0.02365065,-0.02938551,-0.02704978,-0.00671538,0.08657831,0.24425819,0.37283879,0.3645243,0.23266782,0.09534538,0.00949405,-0.02840832,-0.03563838,-0.04079964,-0.04002501,-0.03359789,-0.02736702,-0.02376057,-0.02153566,-0.0265267,-0.03878972,-0.02576192,-0.00063996,-0.00063805,-0.02239153,-0.01758232,-0.02884401,-0.02773998,-0.03199883,-0.02047076,-0.03710848,-0.03795883,-0.03087514,
CAL-22,0,sim_Raman785_demo,1,mixture,calibration,10.0,5.0,85.0,0.18215809,0.18915199,0.18330452,0.18095477,0.17979505,0.18737818,0.1712774,0.1816955,0.19672186,0.17168172,0.17619818,0.19912206,0.24012922,0.22738219,0.20950962,0.17757166,0.18866617,0.18051085,0.20763986,0.23628578,0.20445477,0.21552589,0.32399258,0.58382091,0.73963408,0.59604759,0.34506927,0.21024818,0.19765798,0.20170879,0.39790725,0.58650919,0.83162204,0.83781861,0.5809673,0.32052926,0.20510325,0.1909194,0.19136709,0.18816444,0.18358994,0.21615391,0.29776013,0.43438845,0.56773414,0.57647228,0.44111016,0.29849247,0.21863355,0.1871304,0.1756536,0.17907641,0.18242838,0.18021271,0.18653738,0.17014558,0.18526164,0.17869805,0.17556561,0.18461927,0.20363534,0.20292519,0.18885648,0.18507986,0.1831269,0.18583507,0.16900753,0.18068026,0.17857197,0.18736503,0.17795566,
CAL-22,0,sim_Raman785_demo,2,mixture,calibration,10.0,5.0,85.0,0.18366746,0.17698554,0.18977144,0.18493912,0.18238688,0.17164486,0.19110372,0.18290159,0.18633373,0.18566668,0.18622724,0.21070272,0.23963565,0.22546299,0.20499842,0.17691865,0.18543058,0.18867126,0.2109111,0.25699227,0.21995883,0.21062237,0.34148425,0.59424356,0.75002233,0.59186591,0.33673267,0.21250065,0.18557955,0.20641124,0.39861313,0.59196422,0.83119497,0.85107698,0.58327073,0.3147596,0.19742943,0.18858375,0.18456008,0.18700283,0.19881015,0.21189146,0.29590957,0.44480793,0.57257501,0.58915594,0.45166909,0.29997475,0.23271071,0.19031522,0.19526841,0.1776649,0.18491954,0.1891306,0.19033983,0.19876868,0.1873138,0.18441438,0.18326033,0.18815539,0.21162831,0.21262503,0.18311548,0.18946697,0.18805677,0.17550915,0.18655228,0.18761534,0.18555431,0.19916613,0.18573362,
CAL-23,0,sim_Raman785_demo,1,mixture,calibration,10.0,7.5,82.5,0.00765944,0.00678151,0.00040121,0.00307862,0.0080041,0.00200007,0.00613139,-0.0024087,0.00630986,-0.00039404,0.00482388,0.02332098,0.07159985,0.07882786,0.03456395,0.01008639,0.01450489,-0.00388433,0.046638,0.08982208,0.04817004,0.04104402,0.15335113,0.39519536,0.54450089,0.40341252,0.14279864,0.03615896,0.00563547,0.02967086,0.2292111,0.40284911,0.65138846,0.65802976,0.39667204,0.14601069,0.02938343,0.00375417,0.00801667,0.00826805,0.01936497,0.03185248,0.11756085,0.24417634,0.39438949,0.39231289,0.26858276,0.11886261,0.04486734,0.01608836,0.00031672,0.01142651,0.00711358,0.00549401,0.00661563,0.00225375,0.00956693,-0.01022777,-0.00196393,0.01250959,0.03156616,0.03007006,0.01360002,0.01157096,0.00598155,0.00319665,-0.00226447,0.00450549,0.00250096,0.00400027,0.00801552,
CAL-23,0,sim_Raman785_demo,2,mixture,calibration,10.0,7.5,82.5,0.02699056,0.03035957,0.02479062,0.02056476,0.02010617,0.02209196,0.01513221,0.02173756,0.03156687,0.0265944,0.02947413,0.04836918,0.10248233,0.1150129,0.0471711,0.02466631,0.02790094,0.028357,0.05359925,0.10993782,0.07389421,0.06473631,0.1771604,0.43279823,0.56934906,0.41604429,0.1883025,0.04771144,0.03449148,0.04709637,0.24024523,0.42107075,0.66615117,0.66744395,0.42393161,0.15876396,0.04610728,0.03233074,0.02204115,0.02594063,0.03232364,0.06492682,0.13361807,0.27810862,0.40921658,0.406734,0.28867267,0.14108335,0.07356333,0.03381548,0.0255121,0.03053907,0.01488182,0.01689316,0.02749909,0.03564808,0.02249728,0.02887097,0.02409001,0.03235836,0.05517283,0.05551749,0.03624052,0.02959703,0.03184663,0.02455115,0.01897202,0.02374434,0.02247743,0.03136824,0.0257926,
CAL-24,0,sim_Raman785_demo,1,mixture,calibration,10.0,10.0,80.0,-0.02882393,-0.0312449,-0.03380577,-0.02932756,-0.03043368,-0.02740764,-0.03154551,-0.02790901,-0.03308251,-0.0294373,-0.02436241,0.00069833,0.07043108,0.06070534,0.00041116,-0.02167727,-0.02315996,-0.02004274,-0.00197953,0.07031605,0.02865511,0.01233891,0.117852,0.3592409,0.50492665,0.3545215,0.11672015,-0.00142988,-0.02214225,-0.01816054,0.17894929,0.33768236,0.60187827,0.61247208,0.36445059,0.11222628,0.00104788,-0.03113514,-0.02373498,-0.03019298,-0.02081191,-0.00980233,0.06938243,0.22257237,0.35084536,0.35020961,0.22778711,0.09405378,0.0097587,-0.02255251,-0.0335359,-0.02509236,-0.02967828,-0.02513985,-0.03426856,-0.02301854,-0.02413662,-0.03017643,-0.02935025,-0.02598106,-0.00473006,0.00499812,-0.02565701,-0.02866135,-0.03446334,-0.03051762,-0.03015299,-0.03401885,-0.02949838,-0.02914074,-0.02361045,
CAL-24,0,sim_Raman785_demo,2,mixture,calibration,10.0,10.0,80.0,0.00484893,0.00017175,0.0048956,-0.00366669,-0.00662809,-0.01266646,-0.01247661,0.00217959,-0.00076735,-0.00808558,-0.00048338,0.02664088,0.09796956,0.09239236,0.01428285,-0.00489814,-0.0032756,-0.00050816,0.04301219,0.10674795,0.05303508,0.03575187,0.14348773,0.38102504,0.53064149,0.38043533,0.15128316,0.02369559,0.000355,0.01858399,0.20517148,0.38168461,0.6207422,0.64213176,0.39686307,0.13538407,0.0240451,0.01104877,-0.01905304,-0.00701992,0.01124743,0.03508559,0.09562456,0.23085115,0.36309759,0.35901505,0.25257329,0.10862233,0.03566728,0.00104752,0.0120622,-0.00599968,0.0039824,0.00324011,-0.00693733,0.00208134,0.00116499,0.00354898,0.00441632,0.00355347,0.03357751,0.02502448,0.00028225,-0.00155338,-0.005225,0.00222516,0.01442837,-0.00205203,-0.0017821,0.00747406,0.00381541,
BLIND-00,0,sim_Raman785_demo,1,mixture,blind_test,3.55,3.38,93.07,0.08351574,0.09119782,0.08044268,0.08019692,0.08948835,0.0821882,0.09165,0.09912367,0.08855898,0.10092595,0.0938403,0.09408299,0.12049541,0.11821873,0.09039235,0.0993155,0.08994078,0.09252275,0.11465133,0.13431822,0.10909631,0.13552075,0.25501409,0.53919975,0.68759061,0.5308353,0.24794777,0.12036983,0.09933792,0.1144549,0.25884973,0.50708516,0.79552659,0.7976067,0.51185749,0.23515793,0.11383934,0.07929995,0.08196993,0.08979102,0.09265359,0.1292069,0.21332182,0.3709956,0.51269477,0.50882938,0.37687259,0.21214785,0.12395097,0.10658985,0.0855459,0.09249417,0.0931528,0.10142859,0.09089193,0.08153031,0.0953935,0.07980973,0.09165668,0.08485436,0.09556373,0.11027551,0.08533008,0.0934315,0.08439425,0.08778318,0.10646578,0.09229879,0.09033252,0.09119684,0.09054618,
BLIND-00,0,sim_Raman785_demo,2,mixture,blind_test,3.55,3.38,93.07,0.10610523,0.11076428,0.11137209,0.09521406,0.10211337,0.09950668,0.10695083,0.10739646,0.09977115,0.10777002,0.09792399,0.10682062,0.14016752,0.1432639,0.1131774,0.09025163,0.10943283,0.11067068,0.12168609,0.13973999,0.13154468,0.13232004,0.28229188,0.54615423,0.70916785,0.54391769,0.25941442,0.14796146,0.12021254,0.14122019,0.27212057,0.5220903,0.80615336,0.81242712,0.54520819,0.26206675,0.12673563,0.10505836,0.10454553,0.09542276,0.10470762,0.14364535,0.22806817,0.39187446,0.5267426,0.52781593,0.39530767,0.22479705,0.14712084,0.10304577,0.11387499,0.10355499,0.11401938,0.09565226,0.10546379,0.1084948,0.11017774,0.10504106,0.11875915,0.10455803,0.11756247,0.11266028,0.10401685,0.09937624,0.10763564,0.11770471,0.10626372,0.10704073,0.11237847,0.11219931,0.10013522,
BLIND-01,0,sim_Raman785_demo,1,mixture,blind_test,4.84,1.34,93.82,0.2450422,0.23546912,0.24344645,0.24764377,0.2357325,0.24604166,0.2380791,0.23812439,0.24191746,0.23580619,0.24226455,0.22642073,0.25317014,0.25776102,0.23739416,0.24762207,0.23398877,0.24088716,0.24843501,0.26025166,0.25440508,0.26846356,0.41385824,0.68052579,0.84177341,0.6908872,0.41472278,0.26871054,0.24843948,0.25308872,0.41532929,0.66426735,0.94556681,0.94694431,0.65436393,0.38305595,0.27400853,0.24149049,0.24769118,0.24622922,0.24773444,0.26949216,0.35145654,0.52916802,0.66859799,0.65311444,0.53383591,0.37201172,0.26802098,0.22801882,0.22676147,0.24378337,0.23242665,0.23808088,0.24324705,0.22514245,0.24213491,0.25445873,0.23972763,0.24468025,0.24663146,0.25483908,0.23749096,0.23528503,0.24131963,0.24290371,0.23510267,0.23983511,0.23561718,0.24191166,0.23274471,
BLIND-01,0,sim_Raman785_demo,2,mixture,blind_test,4.84,1.34,93.82,0.26353901,0.26619487,0.2645712,0.25578556,0.27012683,0.26733639,0.26339944,0.26856151,0.26161828,0.26097067,0.26534976,0.26237222,0.2764137,0.28142387,0.26599211,0.27966625,0.26405583,0.26783984,0.27563113,0.29276471,0.27426586,0.29225151,0.42436912,0.70181658,0.87078622,0.70488345,0.42312763,0.287572,0.27383665,0.29387819,0.45307786,0.69680864,0.9843068,0.97268678,0.68440105,0.40350257,0.28270223,0.27056992,0.27117206,0.25847534,0.26667135,0.31283384,0.38758964,0.54361078,0.68802775,0.69048337,0.54780538,0.39874098,0.30415487,0.27668922,0.26453559,0.25578244,0.2526733,0.2734979,0.26688097,0.25856003,0.2669005,0.26838236,0.25320171,0.26839848,0.28396201,0.27867558,0.26669462,0.26560106,0.2685632,0.27721345,0.26077524,0.26339828,0.27183579,0.2692042,0.25927909,
BLIND-02,0,sim_Raman785_demo,1,mixture,blind_test,6.71,5.23,88.06,0.00616372,0.01029479,0.01425509,0.008173,0.01363514,0.01493803,0.01071712,0.00844569,0.01023142,0.02027124,0.01381826,0.01803414,0.056098,0.06046358,0.0138974,0.02304989,0.00170915,0.004758,0.01995815,0.07269608,0.03880803,0.04756005,0.15888359,0.43175042,0.58206641,0.4402038,0.16484637,0.03667655,0.01586259,0.03616346,0.19895668,0.41960422,0.69071941,0.69920709,0.43074086,0.15030623,0.03713067,0.0126959,0.0191877,0.01103557,0.01517049,0.03455977,0.1234492,0.28093146,0.39997282,0.41445766,0.27510818,0.13105629,0.04393011,0.02649421,0.01351101,0.01373902,0.0111391,-0.00390982,0.01213814,0.01333487,0.00577878,0.00588593,0.01300508,0.00823881,0.03206402,0.02846431,0.01639897,0.01577873,0.00477259,0.01296571,0.00066269,0.00666398,0.01459535,0.00998518,0.01906671,
BLIND-02,0,sim_Raman785_demo,2,mixture,blind_test,6.71,5.23,88.06,0.03489523,0.03374779,0.03360211,0.04072834,0.04865627,0.02786474,0.03491229,0.0311435,0.03581213,0.03721882,0.04087231,0.05286551,0.0884103,0.09892353,0.04676361,0.03577326,0.03937488,0.03939378,0.06131917,0.10708322,0.07345587,0.06801402,0.19386378,0.45980254,0.61029215,0.44817236,0.19731201,0.06749068,0.04484444,0.07036165,0.22686401,0.4452802,0.71325343,0.72194506,0.44750441,0.17696828,0.06133083,0.03397035,0.04031554,0.04603677,0.05219493,0.06189285,0.14737966,0.30079355,0.43615573,0.43564828,0.30503731,0.14953667,0.06799817,0.04073457,0.02337685,0.03145195,0.02803897,0.03683726,0.03142175,0.04221384,0.04320465,0.02821587,0.03098325,0.03570626,0.05060255,0.0528409,0.04156028,0.0396158,0.02676101,0.03887773,0.03174876,0.0394663,0.03005121,0.03963905,0.03322475,
BLIND-03,0,sim_Raman785_demo,1,mixture,blind_test,6.22,5.77,88.01,0.04400781,0.04851782,0.04825659,0.05397777,0.04785444,0.05165666,0.03918386,0.05716662,0.05253065,0.04702014,0.03577208,0.06058526,0.10693073,0.11103441,0.06266638,0.05399259,0.04520109,0.05542921,0.07740803,0.10993677,0.07686977,0.08397627,0.20729465,0.46081091,0.6325208,0.47458661,0.20827997,0.07607927,0.05550546,0.06898014,0.24087245,0.4500566,0.72753135,0.73344651,0.47478277,0.18041825,0.08642654,0.0641533,0.0423926,0.04798114,0.05864949,0.06693131,0.17654012,0.31958958,0.45311764,0.45424552,0.31428936,0.17545535,0.09387141,0.05718464,0.05017232,0.0545451,0.06659839,0.04344245,0.05618972,0.04256117,0.04011554,0.04846066,0.04918264,0.04681843,0.07012677,0.07793458,0.0552787,0.04700668,0.04454768,0.0401666,0.04225844,0.04366673,0.05139653,0.0540152,0.05272784,
BLIND-03,0,sim_Raman785_demo,2,mixture,blind_test,6.22,5.77,88.01,0.03198279,0.03597742,0.02716818,0.03288106,0.03654235,0.04701893,0.03203175,0.03514195,0.04553034,0.0344422,0.04104753,0.04769647,0.08871322,0.08252502,0.05515317,0.02779301,0.04626653,0.03188821,0.0661601,0.09636657,0.07049373,0.06284571,0.18474775,0.45142221,0.60457734,0.45190803,0.19174495,0.06680486,0.02584223,0.04993866,0.20924473,0.43728316,0.70593542,0.71255909,0.4440881,0.17308955,0.06507409,0.04170934,0.03045353,0.03945871,0.03306942,0.07161892,0.14580128,0.29614748,0.43288733,0.4379019,0.29625522,0.14689133,0.07107997,0.03774248,0.03678252,0.0314905,0.02924287,0.03210858,0.04165371,0.03281353,0.03476771,0.02952208,0.03133286,0.038589,0.0431887,0.05104558,0.03121801,0.02772584,0.02816393,0.03287998,0.03121374,0.03703465,0.03150474,0.03127804,0.02939515,
BLIND-04,0,sim_Raman785_demo,1,mixture,blind_test,7.73,2.48,89.79,0.12322258,0.1184275,0.12450761,0.11196149,0.11592487,0.10731198,0.11336873,0.11836565,0.11401801,0.11763183,0.12705635,0.12366684,0.14609132,0.14671806,0.1276209,0.12411797,0.11600482,0.12081621,0.12861355,0.16380528,0.13371205,0.15860132,0.27914893,0.54687959,0.70780796,0.55275852,0.2777289,0.14271993,0.134691,0.14964937,0.32729947,0.53731923,0.80544951,0.80789958,0.52731084,0.2625056,0.13986535,0.1260305,0.11381067,0.1185682,0.12438913,0.14592508,0.23722142,0.39615378,0.51829141,0.53505272,0.39141447,0.24341135,0.161105,0.12849793,0.12167832,0.12691566,0.12326331,0.12256102,0.12104399,0.12387692,0.13178036,0.12686716,0.11518719,0.1388462,0.13864533,0.14665051,0.11419933,0.12707843,0.12253148,0.12149742,0.11757466,0.12505153,0.11668398,0.12123925,0.12056955,
BLIND-04,0,sim_Raman785_demo,2,mixture,blind_test,7.73,2.48,89.79,0.11293796,0.11157406,0.10999867,0.11532874,0.12308615,0.11958616,0.1181854,0.11135736,0.10915699,0.12313735,0.12669278,0.12201557,0.13890018,0.13664815,0.12950023,0.11828819,0.11108213,0.10583389,0.13227246,0.14332691,0.13811315,0.13852694,0.27527009,0.54533346,0.69448936,0.52589269,0.26976963,0.15259067,0.11936304,0.13557548,0.31700824,0.53521654,0.79378079,0.80197208,0.52116286,0.24467887,0.1305762,0.11066893,0.12851978,0.12238527,0.11722537,0.14413922,0.22769604,0.38944645,0.52547903,0.52786843,0.3908488,0.24052913,0.15773378,0.11659285,0.11682099,0.10511577,0.11766156,0.11551332,0.10845372,0.10944311,0.12156307,0.11397431,0.11465233,0.12998987,0.14203254,0.13150711,0.11819023,0.11672255,0.11460299,0.12163282,0.11330547,0.1201474,0.11445057,0.10863611,0.10710855,
BLIND-05,0,sim_Raman785_demo,1,mixture,blind_test,7.28,1.69,91.03,0.2068797,0.20587175,0.19747011,0.18871441,0.20376602,0.19761993,0.20442474,0.19362374,0.20184951,0.20382354,0.19801174,0.20256865,0.20747747,0.21522118,0.19921756,0.2005522,0.1972788,0.21095928,0.20631695,0.22574714,0.21182043,0.21804938,0.36231704,0.62372417,0.77948119,0.62481913,0.36725639,0.22699744,0.19443605,0.22398761,0.4031556,0.60721997,0.87837009,0.89839029,0.60673801,0.33024866,0.22768894,0.2070368,0.19545662,0.19090573,0.21019682,0.22426373,0.32136036,0.4841111,0.60811256,0.62390144,0.47712744,0.33069069,0.23531202,0.2094756,0.19294309,0.19145997,0.20803402,0.19860535,0.19957276,0.20581267,0.19160485,0.19867959,0.19666325,0.2045357,0.21597842,0.21909867,0.193584,0.18799262,0.19522886,0.19305715,0.1900649,0.19623525,0.20693173,0.20012449,0.20295298,
BLIND-05,0,sim_Raman785_demo,2,mixture,blind_test,7.28,1.69,91.03,0.14825766,0.14866004,0.15395068,0.1525668,0.14359713,0.14546981,0.15470014,0.15325069,0.1350403,0.15042365,0.14686989,0.15691035,0.16364213,0.17114257,0.15355556,0.15440991,0.1526756,0.15930556,0.15108831,0.17675449,0.17011596,0.18731253,0.31165092,0.58251176,0.73685947,0.57206751,0.30819608,0.17170764,0.15152745,0.17506461,0.34179477,0.56788876,0.83136891,0.8374754,0.56734037,0.28741459,0.1747533,0.14905983,0.14664808,0.14827347,0.15390323,0.17951325,0.26350134,0.41815596,0.56073222,0.54879244,0.41960374,0.26591727,0.17788649,0.15730189,0.1523404,0.15308205,0.14558982,0.14357884,0.14702924,0.14992376,0.14140345,0.15172848,0.14587886,0.15642649,0.16938385,0.16250578,0.15112116,0.14569715,0.14794883,0.14069094,0.14848077,0.14745751,0.1453807,0.15574787,0.16035508,
BLIND-06,0,sim_Raman785_demo,1,mixture,blind_test,4.16,6.14,89.7,-0.05149095,-0.05182982,-0.06193266,-0.05844464,-0.05193221,-0.05034109,-0.04951111,-0.05025968,-0.05343643,-0.04265243,-0.04076896,-0.03547326,0.00635447,0.00348083,-0.03368748,-0.04545624,-0.05252726,-0.06003091,-0.0283028,0.00030908,-0.02946889,-0.02630476,0.09988952,0.37094805,0.52897027,0.38047374,0.1100751,-0.0313881,-0.04789572,-0.02183352,0.1045305,0.35499488,0.63257702,0.63893951,0.36893833,0.10071658,-0.02014335,-0.06228494,-0.05266668,-0.06276505,-0.04777461,-0.01450206,0.05909018,0.20995385,0.35679415,0.34928689,0.21745161,0.07320212,-0.02098066,-0.05160738,-0.0345446,-0.05597155,-0.04606807,-0.05818324,-0.03888955,-0.04703052,-0.05240137,-0.05701453,-0.05312228,-0.05394544,-0.04424217,-0.03148377,-0.05199691,-0.06412287,-0.05582388,-0.05362775,-0.05454974,-0.05542358,-0.05498835,-0.06330024,-0.05257695,
BLIND-06,0,sim_Raman785_demo,2,mixture,blind_test,4.16,6.14,89.7,-0.04631404,-0.04573805,-0.03543137,-0.0402261,-0.03662715,-0.03825367,-0.02847579,-0.03414487,-0.03799983,-0.02148314,-0.03882584,-0.00862218,0.01191626,0.01273091,-0.01670832,-0.02611729,-0.03333527,-0.0269117,-0.01220419,0.01783439,-0.00284622,-0.00032542,0.11529689,0.3978731,0.54671553,0.38523058,0.12456837,-0.00393899,-0.02941017,-0.0152223,0.13099691,0.36109592,0.64109353,0.65823512,0.37874104,0.1110672,-0.01351957,-0.0422983,-0.04485538,-0.03941595,-0.03669738,-0.00703371,0.08539899,0.22103273,0.35879502,0.36916109,0.23318467,0.07803255,-0.00412134,-0.03926159,-0.03425729,-0.04948224,-0.03866686,-0.03900457,-0.04802897,-0.04324961,-0.03295095,-0.04770008,-0.03897585,-0.04131857,-0.01895817,-0.03152267,-0.0358875,-0.03903726,-0.04078936,-0.03000457,-0.04833209,-0.04058889,-0.03286351,-0.03862091,-0.03867568,
LIB-aspartame,0,sim_Raman785_demo,1,library,reference,,,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.97e-06,0.00108621,0.03863369,0.17852647,0.10718248,0.00836045,8.473e-05,1.1e-07,0.0,0.0,0.0,0.0,1.168e-05,0.0403616,1.0,0.17756917,0.00022598,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.7e-07,4.256e-05,0.00140924,0.017168,0.07694165,0.12685533,0.07694165,0.017168,0.00140924,4.256e-05,4.7e-07,0.0,0.0,0.0,4e-08,2.213e-05,0.00240254,0.0546815,0.26087087,0.26087087,0.0546815,0.00240254,2.213e-05,4e-08,0.0,0.0,0.0,0.0,0.0,aspartame
LIB-sucralose,0,sim_Raman785_demo,1,library,reference,,,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1e-08,4.35e-06,0.00060674,0.02463213,0.29096046,1.0,1.0,0.29096046,0.02463545,0.00139428,0.03915483,0.40795739,0.89106083,0.40795738,0.03915048,0.00078754,3.32e-06,0.0,0.0,0.0,0.0,1.5e-07,1.384e-05,0.00057058,0.01029303,0.08125422,0.28069038,0.42431468,0.28069038,0.08125422,0.01029303,0.00057058,1.384e-05,1.5e-07,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,sucralose
LIB-erythritol,0,sim_Raman785_demo,1,library,reference,,,,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685657,0.10685932,0.10698359,0.10974823,0.13972117,0.29337093,0.63542644,0.85485228,0.6354265,0.29337451,0.13984598,0.11215332,0.13263122,0.25820997,0.60110152,1.0,1.0,0.60110149,0.25820718,0.13250423,0.10926362,0.10702603,0.10750332,0.11279194,0.14197415,0.2400811,0.43091536,0.61226629,0.61226629,0.43091536,0.2400811,0.14197415,0.11279188,0.10749974,0.10690123,0.10685853,0.10685659,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,0.10685654,erythritol
COMMERCIAL-01,0,sim_Raman785_demo,1,unknown,query,1.8,,,-0.00264946,0.02642914,0.1706162,0.38813257,0.52798646,0.3869564,0.15635204,0.01783739,0.00461116,-0.01193187,-0.00768613,-0.00593969,-0.00242757,-0.00663515,-0.02063198,-0.00474597,-0.01124133,-0.00714818,-0.00538601,-0.0017926,-0.00452138,-0.00186792,-0.00639433,-0.00978249,-0.00095391,-0.01472494,-0.00859546,-0.00720419,-0.00878965,0.00574894,0.00717579,-0.00227386,0.01371166,0.1231907,0.36487963,0.68869301,0.82404748,0.68543234,0.37559404,0.11514745,0.02284784,-0.00665675,0.00428575,-0.00684713,-0.00532926,-0.00975884,0.00387573,-0.00453683,-0.00936702,-0.00675347,-0.00444044,-0.00905437,-0.00123308,-0.01248532,-0.01159593,-0.00032135,-0.00653175,-0.01186557,-0.00510173,-0.00628677,0.00266031,0.00053715,0.0003728,-0.00099981,-0.00637389,-0.00634741,-0.00613428,-0.00881864,-0.00387421,-0.00918235,-0.00428789,
"""

In [ ]:
df = load_spectra()
meta, X_all, wn = split_spectra(df)
print(f"讀入 {len(df)} 筆光譜，{wn.min():.0f}-{wn.max():.0f} cm-1，{len(wn)} 個位移點。")
print(meta["role"].value_counts())
df.head()

### （選用）在 Colab 上傳班級實際資料

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_spectra(fname)
#     meta, X_all, wn = split_spectra(df)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_spectra('本機路徑或URL')")

## 2. 校正集設計與正交性檢查

5×5 全因子設計：阿斯巴甜與三氯蔗糖各取 5 個水準（0, 2.5, 5, 7.5, 10%），共 25 個配方組合，赤藻糖醇補足到 100%。理論上這個設計會讓兩個目標**完全不相關**——合併資料後第一件事就是驗證這一點：如果算出來明顯不是 0，代表資料合併或配方紀錄可能有誤植。

每個配方另外重複量測 `N_REP` 次（重新裝樣再測一次），交叉驗證要用 `sample_id` 分組，避免同一配方的重複同時落在訓練與驗證兩邊。

In [ ]:
cal_mask = (meta["split"] == "calibration").to_numpy()
blind_mask = (meta["split"] == "blind_test").to_numpy()
lib_mask = (meta["role"] == "library").to_numpy()
unk_mask = (meta["role"] == "unknown").to_numpy()

cal_meta = meta[cal_mask].reset_index(drop=True)
X_cal_raw = X_all[cal_mask]
Y_asp_cal = cal_meta["aspartame_pct"].to_numpy()
Y_suc_cal = cal_meta["sucralose_pct"].to_numpy()
g_cal = cal_meta["sample_id"].to_numpy()

# corr() computed on the UNIQUE design points (not the replicated rows) --
# duplicating each point N_REP times doesn't change the correlation, but
# computing it on the unique table is clearer pedagogically.
design_table = cal_meta.drop_duplicates("sample_id")[["aspartame_pct", "sucralose_pct"]]
corr_check = float(np.corrcoef(design_table["aspartame_pct"], design_table["sucralose_pct"])[0, 1])
print(f"corr(Y_aspartame, Y_sucralose) = {corr_check:.2e}")
assert abs(corr_check) < 1e-9, "設計應該完全正交，若不是 0 要回頭檢查資料合併/配方紀錄"
print("檢查通過：設計完全正交。")

## 3. 看幾條光譜

阿斯巴甜濃度越高，~1003 cm⁻¹（苯環呼吸振動）附近訊號越明顯；三氯蔗糖濃度越高，600–850 cm⁻¹（C–Cl 區）附近訊號越明顯。

In [ ]:
show_pos = [0, 6, 12, 18, 24]  # a spread of calibration points (rep 1 of each)
show_rows = cal_meta[cal_meta["rep"] == 1].iloc[show_pos] if "rep" in cal_meta else cal_meta.iloc[show_pos]
show_idx = show_rows.index.to_numpy()
labels = [f"asp={r.aspartame_pct:.1f}%,suc={r.sucralose_pct:.1f}%" for r in show_rows.itertuples()]
ax = plot_spectra(wn, X_cal_raw[show_idx], labels=labels, xlabel="Raman shift (cm-1)")
ax.set_title("Example calibration spectra")
plt.tight_layout()
plt.show()

## 4. 前處理比較與分組交叉驗證選 LV

比較「raw（僅置中）」與「基線扣除＋面積歸一化」兩種前處理，用 **`LeaveOneGroupOut`（以 `sample_id` 分組）** 交叉驗證比較 RMSECV（以阿斯巴甜模型為例），避免同一配方的重複測量同時出現在訓練與驗證折中。

In [ ]:
def preprocess_baseline_norm(X):
    X = np.atleast_2d(X)
    baseline = (X[:, :5].mean(axis=1, keepdims=True) + X[:, -5:].mean(axis=1, keepdims=True)) / 2.0
    Xc = X - baseline
    norms = np.sum(np.abs(Xc), axis=1, keepdims=True)
    return Xc / norms


def rmsecv_grouped(X, y, groups, max_lv=8):
    logo = LeaveOneGroupOut()
    out = []
    for lv in range(1, max_lv + 1):
        preds = np.zeros_like(y, dtype=float)
        for tr, va in logo.split(X, y, groups):
            m = PLSRegression(n_components=lv).fit(X[tr], y[tr])
            preds[va] = m.predict(X[va]).ravel()
        out.append(float(np.sqrt(np.mean((preds - y) ** 2))))
    return out


MAX_LV = 8
rmsecv_raw_asp = rmsecv_grouped(X_cal_raw, Y_asp_cal, g_cal, MAX_LV)
X_cal_pre = preprocess_baseline_norm(X_cal_raw)
rmsecv_pre_asp = rmsecv_grouped(X_cal_pre, Y_asp_cal, g_cal, MAX_LV)

print("RMSECV(阿斯巴甜) raw:          ", [round(v, 4) for v in rmsecv_raw_asp])
print("RMSECV(阿斯巴甜) baseline+norm:", [round(v, 4) for v in rmsecv_pre_asp])

use_pre = min(rmsecv_pre_asp) < min(rmsecv_raw_asp)
PREP_NAME = "基線扣除+面積歸一化" if use_pre else "raw（僅置中，PLS內建）"
print("依 RMSECV 選出的前處理：", PREP_NAME)

X_cal = X_cal_pre if use_pre else X_cal_raw
rmsecv_asp = rmsecv_pre_asp if use_pre else rmsecv_raw_asp
best_lv_asp = int(np.argmin(rmsecv_asp)) + 1

rmsecv_suc = rmsecv_grouped(X_cal, Y_suc_cal, g_cal, MAX_LV)
best_lv_suc = int(np.argmin(rmsecv_suc)) + 1
print(f"best LV(阿斯巴甜)={best_lv_asp}   best LV(三氯蔗糖)={best_lv_suc}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(range(1, MAX_LV + 1), rmsecv_raw_asp, "o-", label="raw")
ax.plot(range(1, MAX_LV + 1), rmsecv_pre_asp, "s-", label="baseline + area-normalize")
ax.axvline(best_lv_asp, color="grey", linestyle="--", linewidth=1)
ax.set_xlabel("Number of latent variables (LV)")
ax.set_ylabel("RMSECV (%), aspartame")
ax.set_title("Preprocessing comparison (grouped LeaveOneGroupOut CV)")
ax.legend()
plt.tight_layout()
plt.show()

## 5. 最終 PLS1 模型與盲樣 RMSEP

用選定的前處理與 LV 數，在整個校正集上配適最終模型，然後對 7 個離網格盲樣做**唯一一次**預測，計算 RMSEC、RMSECV、RMSEP、bias、RPD。

In [ ]:
blind_meta = meta[blind_mask].reset_index(drop=True)
X_blind_raw = X_all[blind_mask]
X_blind = preprocess_baseline_norm(X_blind_raw) if use_pre else X_blind_raw
Y_asp_blind = blind_meta["aspartame_pct"].to_numpy()
Y_suc_blind = blind_meta["sucralose_pct"].to_numpy()

pls_asp = PLSRegression(n_components=best_lv_asp).fit(X_cal, Y_asp_cal)
pls_suc = PLSRegression(n_components=best_lv_suc).fit(X_cal, Y_suc_cal)

logo = LeaveOneGroupOut()
cv_pred_asp = np.zeros_like(Y_asp_cal, dtype=float)
for tr, va in logo.split(X_cal, Y_asp_cal, g_cal):
    m = PLSRegression(n_components=best_lv_asp).fit(X_cal[tr], Y_asp_cal[tr])
    cv_pred_asp[va] = m.predict(X_cal[va]).ravel()
cv_pred_suc = np.zeros_like(Y_suc_cal, dtype=float)
for tr, va in logo.split(X_cal, Y_suc_cal, g_cal):
    m = PLSRegression(n_components=best_lv_suc).fit(X_cal[tr], Y_suc_cal[tr])
    cv_pred_suc[va] = m.predict(X_cal[va]).ravel()

pred_blind_asp = pls_asp.predict(X_blind).ravel()
pred_blind_suc = pls_suc.predict(X_blind).ravel()


def figures_of_merit(y_cal, model, X_cal_, cv_pred, y_blind, pred_blind):
    rmsec = float(np.sqrt(np.mean((model.predict(X_cal_).ravel() - y_cal) ** 2)))
    rmsecv = float(np.sqrt(np.mean((cv_pred - y_cal) ** 2)))
    rmsep = float(np.sqrt(np.mean((pred_blind - y_blind) ** 2)))
    bias = float(np.mean(pred_blind - y_blind))
    rpd = float(np.std(y_blind, ddof=1) / rmsep)
    return rmsec, rmsecv, rmsep, bias, rpd


rmsec_asp, rmsecv_asp_final, rmsep_asp, bias_asp, rpd_asp = figures_of_merit(
    Y_asp_cal, pls_asp, X_cal, cv_pred_asp, Y_asp_blind, pred_blind_asp)
rmsec_suc, rmsecv_suc_final, rmsep_suc, bias_suc, rpd_suc = figures_of_merit(
    Y_suc_cal, pls_suc, X_cal, cv_pred_suc, Y_suc_blind, pred_blind_suc)

fom = pd.DataFrame([
    {"sweetener": "aspartame", "LV": best_lv_asp, "RMSEC": rmsec_asp, "RMSECV": rmsecv_asp_final,
     "RMSEP": rmsep_asp, "bias": bias_asp, "RPD": rpd_asp},
    {"sweetener": "sucralose", "LV": best_lv_suc, "RMSEC": rmsec_suc, "RMSECV": rmsecv_suc_final,
     "RMSEP": rmsep_suc, "bias": bias_suc, "RPD": rpd_suc},
])
fom

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6.5))
ax.scatter(Y_asp_cal, cv_pred_asp, s=40, color="tab:blue", alpha=0.7, label="Calibration (grouped CV)")
ax.scatter(Y_asp_blind, pred_blind_asp, s=80, color="tab:orange", marker="^", label="Blind blends")
lo, hi = -1, 11
ax.plot([lo, hi], [lo, hi], "--", color="grey", label="1:1 line")
ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
ax.set_xlabel("Actual aspartame (%)")
ax.set_ylabel("PLS-predicted aspartame (%)")
ax.set_title("Aspartame PLS1: predicted vs actual")
ax.legend()
ax.set_aspect("equal", adjustable="box")
plt.tight_layout()
plt.show()

## 6. VIP vs Raman Shift

跟 NB12 用完全一樣的標準公式計算 VIP，對照官能基指認：阿斯巴甜的 VIP 應該在 **~1003 cm⁻¹**（苯環呼吸振動，這是文獻上較有共識的歸屬）附近最高；三氯蔗糖的 VIP 應該在 **600–850 cm⁻¹**（C–Cl 伸縮振動區）附近偏高——**這個範圍的歸屬做了保守處理**，實際文獻上對三氯蔗糖 C–Cl 相關峰的確切波數仍有一定分歧，這裡取一個較寬的區間，並不代表 850 cm⁻¹ 以內每一點都對應到某個特定的鍵振動。

In [ ]:
def compute_vip(pls):
    T = pls.x_scores_
    W = pls.x_weights_
    Q = pls.y_loadings_
    p, ncomp = W.shape
    SSY = np.array([(Q[0, a] ** 2) * np.sum(T[:, a] ** 2) for a in range(ncomp)])
    Wn = W / np.linalg.norm(W, axis=0, keepdims=True)
    return np.sqrt(p * (Wn ** 2 @ SSY) / SSY.sum())


vip_asp = compute_vip(pls_asp)
vip_suc = compute_vip(pls_suc)

print(f"mean(VIP_asp^2) = {np.mean(vip_asp**2):.9f}   mean(VIP_suc^2) = {np.mean(vip_suc**2):.9f}")
assert abs(np.mean(vip_asp ** 2) - 1.0) < 1e-9
assert abs(np.mean(vip_suc ** 2) - 1.0) < 1e-9

idx_1003 = int(np.argmin(np.abs(wn - 1003)))
peak_idx_asp = int(np.argmax(vip_asp))
print(f"VIP(阿斯巴甜) 全域最大值在 {wn[peak_idx_asp]:.0f} cm-1 (最接近1003的格點是 {wn[idx_1003]:.0f} cm-1)")
assert peak_idx_asp == idx_1003, "阿斯巴甜模型的最高 VIP 應該落在 1003 cm-1 附近"

c_cl_mask = (wn >= 600) & (wn <= 850)
print(f"VIP(三氯蔗糖) 在 600-850 cm-1 (C-Cl 區) 的最大值 = {vip_suc[c_cl_mask].max():.3f}")
assert vip_suc[c_cl_mask].max() > 1.5, "三氯蔗糖模型應該在 C-Cl 區有明顯高 VIP"
print("三個結構性檢查（mean(VIP^2)=1 x2、VIP高峰對上化學指認）全部通過。")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axes[0].plot(wn, vip_asp, color="tab:blue")
axes[0].axhline(1.0, color="tab:red", linestyle="--", linewidth=1)
axes[0].axvline(1003, color="green", linestyle=":", linewidth=1.2, label="1003 cm-1 (aspartame ring breathing)")
axes[0].set_ylabel("VIP (aspartame model)")
axes[0].legend(fontsize=8)

axes[1].plot(wn, vip_suc, color="tab:purple")
axes[1].axhline(1.0, color="tab:red", linestyle="--", linewidth=1)
axes[1].axvspan(600, 850, color="orange", alpha=0.15, label="600-850 cm-1 (C-Cl region)")
axes[1].set_xlabel("Raman shift (cm-1)")
axes[1].set_ylabel("VIP (sucralose model)")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

ery_bands = [880, 1050, 1290]
ery_idx = [int(np.argmin(np.abs(wn - w))) for w in ery_bands]
print("VIP(阿斯巴甜模型) 在赤藻糖醇填充劑波段 (880/1050/1290 cm-1):", np.round(vip_asp[ery_idx], 3))

**封閉性（closure）問題**：因為三個成分的比例加總永遠是 100%，赤藻糖醇的波段也可能出現 VIP > 1——阿斯巴甜或三氯蔗糖增加時，赤藻糖醇一定跟著減少，填充劑波段因此帶有「間接」的預測力。但這種間接關聯**不具特異性**：換一批用不同填充劑（例如市售代糖常見的糊精）的樣品，這個關聯就會失效，第 9 節會實際示範這個狀況。

## 7. NNLS／古典最小平方（不需要校正集的另一條路徑）

只用**純物質參考光譜**（`role == "library"` 的三筆光譜）做非負最小平方擬合：

$$\text{未知光譜} \approx c_1 \cdot \text{阿斯巴甜參考} + c_2 \cdot \text{三氯蔗糖參考} + c_3 \cdot \text{赤藻糖醇參考} + \text{offset}, \quad c_1, c_2, c_3 \ge 0$$

不需要另外準備校正集，$c$ 值直接就是重量分率，物理意義清楚；缺點是假設**線性疊加**，遇到偵測器飽和、基質效應等非線性時，比例估計會有系統性偏誤。

In [ ]:
lib_meta = meta[lib_mask].reset_index(drop=True)
lib_spec = {row.loc["class"]: X_all[lib_mask][i] for i, (_, row) in enumerate(lib_meta.iterrows())}
A_ref = np.column_stack([lib_spec["aspartame"], lib_spec["sucralose"], lib_spec["erythritol"], np.ones(len(wn))])


def nnls_fit(spec):
    coef, _ = nnls(A_ref, spec)
    return coef[0] * 100, coef[1] * 100, coef[2] * 100, coef[3]


nnls_cal = np.array([nnls_fit(x) for x in X_cal_raw])
nnls_blind = np.array([nnls_fit(x) for x in X_blind_raw])

rmsec_nnls_asp = float(np.sqrt(np.mean((nnls_cal[:, 0] - Y_asp_cal) ** 2)))
rmsep_nnls_asp = float(np.sqrt(np.mean((nnls_blind[:, 0] - Y_asp_blind) ** 2)))
rmsec_nnls_suc = float(np.sqrt(np.mean((nnls_cal[:, 1] - Y_suc_cal) ** 2)))
rmsep_nnls_suc = float(np.sqrt(np.mean((nnls_blind[:, 1] - Y_suc_blind) ** 2)))

method_compare = pd.DataFrame([
    {"method": f"PLS1 (LV={best_lv_asp}/{best_lv_suc}, {PREP_NAME})", "RMSEP_aspartame": rmsep_asp, "RMSEP_sucralose": rmsep_suc},
    {"method": "NNLS/CLS (3 components + offset)", "RMSEP_aspartame": rmsep_nnls_asp, "RMSEP_sucralose": rmsep_nnls_suc},
])
method_compare

PLS 在校正時已經「看過」偵測器輕微飽和等真實變異，通常 RMSEP 較小；NNLS 假設嚴格線性疊加，遇到這些變異時誤差較大——但換一批完全沒訓練過的新成分時（例如第 9 節的市售樣品），NNLS 不會亂外插，反而可能比 PLS「誠實」。

## 8. 近似 LOD（教學簡化法，非正式方法確效）

用交叉驗證預測值裡「0% 水準」的標準差，除以「預測值 vs 實際值」迴歸的斜率，估計一個近似的 LOD／LOQ：

$$\mathrm{LOD} \approx \dfrac{3.3\,\sigma_0}{\text{slope}}, \qquad \mathrm{LOQ} \approx \dfrac{10\,\sigma_0}{\text{slope}}$$

> ⚠️ 這是**教學上的簡化估計法**，不是正式的方法確效（method validation）流程（正式做法需要多次獨立重複量測空白/低濃度樣品，並考慮更嚴謹的統計假設）。

In [ ]:
def approx_lod(y_design, cv_pred):
    mask0 = (y_design == 0.0)
    sd0 = float(np.std(cv_pred[mask0], ddof=1))
    slope, intercept = np.polyfit(y_design, cv_pred, 1)
    lod = 3.3 * sd0 / slope
    loq = 10 * sd0 / slope
    return lod, loq, sd0, slope


LOD_asp, LOQ_asp, sd0_asp, slope_asp = approx_lod(Y_asp_cal, cv_pred_asp)
LOD_suc, LOQ_suc, sd0_suc, slope_suc = approx_lod(Y_suc_cal, cv_pred_suc)

print(f"阿斯巴甜: slope={slope_asp:.4f}  sd(0%)={sd0_asp:.4f}  LOD≈{LOD_asp:.3f}%  LOQ≈{LOQ_asp:.3f}%")
print(f"三氯蔗糖  : slope={slope_suc:.4f}  sd(0%)={sd0_suc:.4f}  LOD≈{LOD_suc:.3f}%  LOQ≈{LOQ_suc:.3f}%")

## 9. 市售樣品的適用範圍檢查（T²／Q）

跟 NB12 用一樣的方法（PCA + Hotelling T² + Jackson–Mudholkar Q 界限）檢查一個「市售代糖」樣品是否落在模型的**適用範圍**內。這包代糖標示阿斯巴甜 1.8%，但填充劑是**糊精**（dextrose-like），跟校正集用的**赤藻糖醇**完全不同。

In [ ]:
commercial_asp_true = float(meta.loc[unk_mask, "aspartame_pct"].iloc[0])  # declared label value

pca_full = PCA(n_components=min(len(X_cal) - 1, X_cal.shape[1] - 1)).fit(X_cal)
eigvals_all = pca_full.explained_variance_

NCOMP_PCA = 3
pca = PCA(n_components=NCOMP_PCA).fit(X_cal)
eigvals = pca.explained_variance_[:NCOMP_PCA]

n_cal_pts = X_cal.shape[0]
F_crit = stats.f.ppf(0.95, NCOMP_PCA, n_cal_pts - NCOMP_PCA)
T2_limit = NCOMP_PCA * (n_cal_pts - 1) / (n_cal_pts - NCOMP_PCA) * F_crit

theta1 = float(np.sum(eigvals_all[NCOMP_PCA:]))
theta2 = float(np.sum(eigvals_all[NCOMP_PCA:] ** 2))
theta3 = float(np.sum(eigvals_all[NCOMP_PCA:] ** 3))
h0 = 1 - (2 * theta1 * theta3) / (3 * theta2 ** 2)
z95 = stats.norm.ppf(0.95)
Q_limit = theta1 * (z95 * np.sqrt(2 * theta2 * h0 ** 2) / theta1 + 1 + theta2 * h0 * (h0 - 1) / theta1 ** 2) ** (1 / h0)
print(f"T2_limit (95%) = {T2_limit:.3f}   Q_limit (95%) = {Q_limit:.6f}")

comm_raw = X_all[unk_mask][0]
comm_x = preprocess_baseline_norm(comm_raw.reshape(1, -1))[0] if use_pre else comm_raw

pred_comm_asp = float(np.ravel(pls_asp.predict(comm_x.reshape(1, -1)))[0])
T_comm = pca.transform(comm_x.reshape(1, -1))
T2_comm = float(np.sum((T_comm ** 2) / eigvals))
recon_comm = pca.inverse_transform(T_comm)
Q_comm = float(np.sum((comm_x - recon_comm) ** 2))

print(f"\n標示阿斯巴甜 = {commercial_asp_true}%   PLS 直接預測 = {pred_comm_asp:.2f}%")
print(f"T2={T2_comm:.2f} (界限{T2_limit:.2f}, {'超出' if T2_comm > T2_limit else '正常'})   "
      f"Q={Q_comm:.5f} (界限{Q_limit:.5f}, {'超出' if Q_comm > Q_limit else '正常'})")

assert Q_comm > Q_limit, "市售樣品使用了校正集沒見過的填充劑，Q 殘差應明顯超標"
print("\n檢查通過：Q 殘差超標，代表這包代糖『長得不像』校正集學過的任何東西——"
      "PLS 的預測值不能直接採信，即使數字本身看起來『合理』。")

**誠實的報告方式**：先看 Q／T²，超出適用範圍就不能直接報數字，只能寫「超出模型適用範圍，建議重新採集校正集或改用其他方法」；若 Q／T² 正常但預測值低於 LOD，則報告「< LOD」，而不是硬報一個看起來精確、實際上沒有意義的數字。

## 10. W17 模型報告卡

In [ ]:
# Build the qualitative statements FROM the actual numbers computed above,
# rather than hardcoding an assumed outcome -- so a rerun on different/real
# data can't silently print a claim the numbers no longer support.
asp_peak_ok = peak_idx_asp == idx_1003
suc_ccl_ok = vip_suc[c_cl_mask].max() > 1.5
vip_match_str = (
    f"阿斯巴甜峰值在{wn[peak_idx_asp]:.0f}cm-1({'符合1003cm-1' if asp_peak_ok else '不符合1003cm-1，需檢查'})；"
    f"三氯蔗糖C-Cl區VIP最大值={vip_suc[c_cl_mask].max():.2f}"
    f"({'符合(>1.5)' if suc_ccl_ok else '不符合，需檢查'})"
)
nnls_compare_str = (
    f"PLS較優 ({rmsep_asp:.2f} vs {rmsep_nnls_asp:.2f}% 阿斯巴甜)" if rmsep_asp < rmsep_nnls_asp
    else f"NNLS較優 ({rmsep_nnls_asp:.2f} vs {rmsep_asp:.2f}% 阿斯巴甜)"
)
commercial_result_str = "超出適用範圍 (Q超標)，未報數值" if Q_comm > Q_limit else "在適用範圍內"

report_card = pd.DataFrame([
    {"item": "儀器", "value": "sim_Raman785_demo"},
    {"item": "前處理", "value": PREP_NAME},
    {"item": "LV (阿斯巴甜/三氯蔗糖)", "value": f"{best_lv_asp}/{best_lv_suc}"},
    {"item": "RMSEC/RMSECV/RMSEP (阿斯巴甜, %)", "value": f"{rmsec_asp:.2f}/{rmsecv_asp_final:.2f}/{rmsep_asp:.2f}"},
    {"item": "RMSEC/RMSECV/RMSEP (三氯蔗糖, %)", "value": f"{rmsec_suc:.2f}/{rmsecv_suc_final:.2f}/{rmsep_suc:.2f}"},
    {"item": "bias/RPD (阿斯巴甜)", "value": f"{bias_asp:+.2f}% / {rpd_asp:.2f}"},
    {"item": "LOD (阿斯巴甜/三氯蔗糖)", "value": f"{LOD_asp:.2f}% / {LOD_suc:.2f}%"},
    {"item": "VIP-波段吻合度", "value": vip_match_str},
    {"item": "NNLS 比較", "value": nnls_compare_str},
    {"item": "商用樣品結果", "value": commercial_result_str},
])
report_card

## 11. 練習題 (Exercises)

### 問題 1

把第 2 節的正交性檢查改成故意打亂三氯蔗糖那一欄的順序（例如 `np.random.permutation`）後再算一次 `corr()`。相關係數還會接近 0 嗎？這個練習想說明什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 2（資料外洩示範）

把第 4 節的 `LeaveOneGroupOut`（以 `sample_id` 分組）換成不分組的普通 `KFold`，對阿斯巴甜模型重新算一次 RMSECV（固定用 `best_lv_asp`）。跟分組結果比較，哪一個比較樂觀？為什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 3

對照第 6 節的 VIP 圖與封閉性（closure）說明：如果今天赤藻糖醇換成一個化學結構跟阿斯巴甜、三氯蔗糖完全不重疊的填充劑，你預期赤藻糖醇波段的 VIP 會比現在更高、更低，還是差不多？為什麼？

In [ ]:
# TODO: 請在這裡作答

### 問題 4

第 7 節 NNLS 對盲樣的 RMSEP 比 PLS 差。試著解釋：如果今天完全沒有校正集（沒有 25 個已知配方），只有三條純物質參考光譜，你還能用 PLS 嗎？為什麼 NNLS 在這種情況下是唯一可行的選擇？

In [ ]:
# TODO: 請在這裡作答

### 問題 5

第 9 節的市售樣品，PLS 直接預測出一個數字，但 Q 殘差超標。如果今天 Q／T² 都正常、但預測值低於第 8 節算出的 LOD，你應該在報告中怎麼寫這個樣品的分析結果？

In [ ]:
# TODO: 請在這裡作答

## 12. 匯出結果 (Export)

In [ ]:
output_path = "nb13_model_report_card.csv"
report_card.to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

try:
    from google.colab import files
    files.download(output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")